<p class="lead">This notebook is the R edition of the second unit of the course. The lecture slides are a map; this text is the territory. Unit 1 ended with a classifier "trained" by computing two means; this unit asks what training means in general — what it is for a machine to <em>learn</em> — and answers as a definition (Section 1), as a step in the practice of data science (Section 2), as a family of paradigms (Sections 3–5) and as an algorithm you can run by hand (Sections 6–7). The official plan omits regression, the simplest learning problem and the one in which model, loss, optimisation and overfitting can be seen without distraction; regression is therefore the core of this unit, and logistic regression — a regression model used as a classifier — is the first classifier studied properly. Every figure and number in the text is produced by the R code you can see and change.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>State Mitchell's definition of learning, identify task, performance and experience in a clinical problem, and distinguish inductive from deductive inference.</li>
<li>Define data science, structured and unstructured data and the seven steps of a project, and locate machine learning within them.</li>
<li>Distinguish supervised, unsupervised, reinforcement, semi- and self-supervised learning with a biomedical example of each; use "class", "label" and "target" correctly.</li>
<li>Define a classifier and its decision boundary, and relate the statistical, neural and evolutionary approaches to Unit 1.</li>
<li>Derive and solve the normal equations, interpret coefficients, \(R^2\) and residuals; implement gradient descent and explain the learning rate; recognise overfitting and explain ridge.</li>
<li>Explain the sigmoid, log-loss and threshold of logistic regression, compute one probability by hand, and extend to several classes.</li>
<li>Say what k-means and value iteration learn, and from what.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read linearly once, executing each code chunk and reading the paragraph that interprets it; then return by section. Green boxes are exercises that change one thing in the chunk above; blue boxes key ideas; orange boxes misconceptions; purple boxes links to other units or to BGM324. Only base R and the packages <code>MASS</code>, <code>class</code>, <code>e1071</code>, <code>rpart</code>, <code>rpart.plot</code>, <code>nnet</code>, <code>cluster</code>, <code>mlbench</code> and <code>ggplot2</code> are used; in a fresh Google Colab R runtime run once <code>install.packages(c("e1071","rpart.plot","nnet","cluster","mlbench"))</code>. The datasets ship with <code>mlbench</code> or with base R, so nothing is downloaded. Supplement S0 covers the R toolkit, Supplement S1 the evaluation methodology previewed in Section 11.</p></div>



In [ ]:
# Setup: packages used throughout, and a fixed seed so every quoted number is reproducible.
suppressPackageStartupMessages({
  library(MASS); library(class); library(e1071); library(rpart); library(rpart.plot)
  library(nnet); library(cluster); library(mlbench); library(ggplot2)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
options(digits = 5)
R.version.string



The printout is the R version these notes were knitted with. R is a language built for data — statistics, modelling and graphics are in the language rather than bolted on<sup class="c"><a href="#r36">[36]</a></sup> — and it is the language the official laboratory plan uses, so these notebooks double as lab material; the R companion to the whole course is *An Introduction to Statistical Learning with Applications in R*.<sup class="c"><a href="#r39">[39]</a></sup>

## 1. What it means to learn

### 1.1 Mitchell's definition

The field's operational definition of learning is Tom Mitchell's:<sup class="c"><a href="#r1">[1]</a></sup>

<div class="box key"><h4>Definition (Mitchell, 1997)</h4><p>A computer program is said to <strong>learn</strong> from experience <em>E</em> with respect to some class of tasks <em>T</em> and performance measure <em>P</em>, if its performance at tasks in <em>T</em>, as measured by <em>P</em>, improves with experience <em>E</em>.</p></div>

It is purely behavioural, and it turns "does the system learn?" into three concrete questions every application paper should answer.

| Component | Question | Breast-cancer example (Unit 1) | Hospital-ward example |
|---|---|---|---|
| **Task T** | What is the system to do? | Assign <em>malignant</em> or <em>benign</em> to an aspirate described by nuclear features | Predict whether a patient will deteriorate within 24 h |
| **Performance P** | How is doing well measured? | Agreement with the biopsy (better: the two error rates separately) | Sensitivity at a fixed false-alarm rate |
| **Experience E** | What data does it see? | 683 labelled aspirates from Wisconsin | Vital signs and outcomes of past admissions |

The idea predates the definition. In 1959 Arthur Samuel's checkers program improved by playing against itself and adjusting the weights of its evaluation function; he called it "machine learning": programming computers to learn from experience so as to eliminate much of the detailed programming.<sup class="c"><a href="#r2">[2]</a></sup> The shift from expert systems to learned systems (Unit 1) is the shift from writing the rule to writing the <em>procedure that finds the rule</em>.

### 1.2 The three ingredients of a learning algorithm

Every learning algorithm in this course is built from three parts:

1. **A model (hypothesis space):** a family of functions \(f_\theta\) indexed by parameters \(\theta\) — intercept and slope for a line, millions of weights for a network; the algorithm can only find a member of the family.
2. **A loss function** \(L(\theta)\): how badly \(f_\theta\) performs on the training data — squared errors, misclassifications, negative log-probability of the labels; it operationalises Mitchell's \(P\).
3. **An optimiser:** the search for the parameters minimising the loss — a closed formula (Section 6), iterative descent (Section 7) or evolutionary search (Section 4, Unit 7).

Domingos compresses this into learning = representation + evaluation + optimisation; his dozen further lessons — generalisation is what counts, data alone are not enough, overfitting has many faces — this unit illustrates one by one.<sup class="c"><a href="#r3">[3]</a></sup>

<div class="box warn"><h4>Common misconception</h4><p>"Training" is not "programming the answer". The human supplies the form (model), the criterion (loss), the data and the labels; the algorithm finds the parameters. When a model "was trained on 100,000 X-rays", nobody wrote the rule that reads the X-ray, and nobody may be able to read it.</p></div>

### 1.3 Inductive and deductive inference

Learning in Mitchell's sense is <strong>inductive inference</strong>: from particular observations to a general rule. <strong>Deductive inference</strong> goes the other way, and the two differ in every respect that matters:

| | Deductive | Inductive |
|---|---|---|
| **Direction** | general → specific: rules to cases | specific → general: cases to a rule |
| **Role** | <em>apply</em> knowledge someone already has | <em>discover</em> knowledge from data |
| **Guarantee** | conclusion certain if the premises are true | conclusion probable, never certain |
| **Risk** | rigidity: fails on cases the rules did not anticipate | over-generalisation: a rule that fits the sample, not the world |
| **In AI** | expert systems, logic (Unit 1, Section 3.1) | machine learning (this course) |

"All mammals have lungs; a whale is a mammal; therefore a whale has lungs" is deduction: certain, contained in the premises. "Every mammal examined so far has lungs; therefore all mammals have lungs" is induction: how the rule was obtained, fallible in principle and reliable in practice. Rats learn inductively: one that falls ill after a novel bait avoids that bait — and that flavour — thereafter, which is why poisoned baits fail against rats so quickly. A spam filter shows both in software. A <em>deductive</em> filter stores every message marked as spam and blocks exact copies: it never makes a false accusation and is defeated by one changed character. An <em>inductive</em> filter learns which words characterise the marked messages and scores new ones by them: it <em>generalises</em> to messages never seen, and is sometimes wrong. Memorisation is deduction from a table; learning is induction to a rule, and the methodology of this course exists because induction is fallible and its reliability must be measured.

## 2. From data science to machine learning

### 2.1 What data science is

<strong>Data science</strong> extracts knowledge and actionable conclusions from data by combining statistics, computing and domain knowledge; its roots are in statistics, and Donoho's historical account asks what is genuinely new in it.<sup class="c"><a href="#r4">[4]</a></sup> Machine learning is the part of data science that builds models which learn from data; data science also includes everything before (asking the question; collecting, cleaning and exploring the data) and after (interpreting and communicating). It follows the <strong>scientific method</strong> — observe, question, collect data designed to answer it, analyse, conclude, communicate so others can check — at a scale classical experimentation did not face. In a real project model fitting takes a small fraction of the time; the rest goes to the data.

### 2.2 Kinds of data and where they come from

<strong>Structured data</strong> conform to a predefined data model — a table with fixed columns and types, as the data matrix of Unit 1 — and live in relational databases queried with <strong>SQL</strong> and, institutionally, in a <strong>data warehouse</strong>, a curated store built for known analyses. <strong>Unstructured data</strong> have no predefined model — clinical notes, images, audio, sequences — and live as files or in <strong>NoSQL</strong> databases and, at scale, in a <strong>data lake</strong> of raw data interpreted only when read. Most hospital data are unstructured, and the pattern-recognition pipeline of Unit 1 is the machinery that turns them into the structured table every method here requires. <strong>Quantitative</strong> data admit arithmetic; <strong>qualitative</strong> data are categories, text or judgements. Analysis methods are named by their product: <strong>classification</strong> (a category), <strong>regression</strong> (a number), <strong>clustering</strong> (groups without labels) and <strong>data mining</strong>, the discovery of patterns such as association rules (Section 3.3).

Data come from <strong>transactional systems</strong> (hospital, pharmacy, laboratory records), <strong>public sources</strong> (statistics, repositories such as UCI), <strong>experiments</strong> and trials, curated <strong>databases</strong> (ChEMBL, DrugBank, UniProt), the <strong>literature</strong> and <strong>patents</strong>, <strong>sensors</strong> (monitors, wearables, imaging) and <strong>media</strong>. Each carries its own biases.

### 2.3 Purposes

Projects <strong>describe</strong> a phenomenon (how many patients were readmitted), <strong>explain</strong> it (which factors are associated), <strong>predict</strong> it (which patients will be readmitted) and, downstream, <strong>reduce cost</strong>, <strong>mitigate risk</strong> and <strong>improve efficiency</strong>. Machine learning serves mainly prediction.

### 2.4 The seven steps of a data-science project

1. **Problem identification.** The question, the unit of analysis, the target, the measure of success, the decision informed — Mitchell's \(T\), \(P\), \(E\).
2. **Data collection.** From the sources above, with a legal basis and documented provenance.
3. **Data cleaning.** Quantify <em>missing data</em> and decide (discard, impute, model); detect <em>outliers</em> and judge them; remove unwanted records and columns; unify structure and format.
4. **Exploratory data analysis (EDA).** Descriptive statistics, <em>univariate</em> and <em>multivariate</em> (correlations, cross-tabulations); hypothesis tests where warranted; above all graphical representation.
5. **Feature engineering.** Attribute selection; <em>class balance</em>; grouping of categories; <em>normalisation</em> (Section 7 shows why); <em>feature correlations</em>, since correlated features destabilise linear models (Section 6.5).
6. **Machine-learning prediction.** Choose, fit, evaluate honestly (Section 11, Supplement S1), iterate back to steps 3–5.
7. **Data storytelling.** Communicate figures, uncertainty and limitations to the people who will act, in their language.

Three chunks illustrate steps 3–5 on the Wisconsin breast-cancer data,<sup class="c"><a href="#r5">[5]</a></sup> shipped by the <code>mlbench</code> package<sup class="c"><a href="#r37">[37]</a></sup> and loaded here for the whole notebook. The shipped version has 683 complete aspirates described by nine nuclear-shape features scored 1–10 and labelled benign or malignant; since those data are clean, the first chunk deletes 5 % of the values at random to illustrate missingness.



In [ ]:
# Step 3, data cleaning: curate the data, inject missingness for illustration, flag outliers by z-score.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                  # drop the Id column
bc <- bc[complete.cases(bc), ]                            # drop the 16 incomplete records
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal -> numeric
y <- ifelse(bc$Class == "benign", 1L, 0L)                 # y: 1 = benign, 0 = malignant (as in Unit 1)
X <- bc[, 1:9]

set.seed(0); miss <- matrix(runif(nrow(X) * 9) < 0.05, nrow(X))   # delete 5% of cells at random
cat(sprintf("cells missing: %.1f%% | rows with >=1 missing: %.1f%% | rows surviving 'drop any NA row': %d of %d\n",
            100*mean(miss), 100*mean(rowSums(miss) > 0), sum(rowSums(miss) == 0), nrow(X)))
z <- scale(X)                                             # z-scores of the clean data
cat(sprintf("values beyond 4 SD: %d in %d patients; most extreme feature '%s' at z = %.1f\n",
            sum(abs(z) > 4), sum(rowSums(abs(z) > 4) > 0),
            names(which.max(apply(abs(z), 2, max))), max(abs(z))))



With 5 % of cells missing at random, about 40 % of rows have a gap and dropping incomplete rows would keep 408 patients of 683 — which is why imputation is normal practice and missingness is quantified <em>per column</em>. The z-score screen flags 14 values beyond four standard deviations in 14 patients; whether they are errors or extreme tumours is for the pathologist to judge.



In [ ]:
# Step 4, exploratory data analysis: class means and the correlation structure of the nine features.
means <- t(sapply(X, function(z) tapply(z, bc$Class, mean)))
ratio <- sort(means[, "malignant"] / means[, "benign"], decreasing = TRUE)
cat("features whose malignant mean is the largest multiple of the benign mean:\n")
print(round(ratio[1:4], 2))

corr <- cor(X)
dfp <- data.frame(f1 = rep(colnames(X), 9), f2 = rep(colnames(X), each = 9), r = as.vector(corr))
ggplot(dfp, aes(f1, f2, fill = r)) + geom_tile() +
  geom_text(aes(label = sprintf("%.2f", r)), size = 2.6,
            colour = ifelse(abs(dfp$r) > 0.6, "white", "black")) +
  scale_fill_gradient2(low = BLUE, mid = "white", high = ORANGE, midpoint = 0, limits = c(-1, 1)) +
  labs(title = "Pearson correlation between the nine nuclear features", x = NULL, y = NULL, fill = "r") +
  theme_minimal(base_size = 9) + theme(axis.text.x = element_text(angle = 60, hjust = 1))



<p class="caption"><b>Figure 1.</b> Pearson correlation between the nine nuclear features, drawn with <code>ggplot2</code>.<sup class="c"><a href="#r40">[40]</a></sup> Cell size and cell shape correlate at 0.91 — two measures of the same nuclear enlargement — so a model given both receives that information twice (Section 6.5 shows the consequence). The table shows that bare nuclei, cell size and normal nucleoli differ most between classes (malignant means about five times the benign means), which is why the two features used from Unit 1 onward are clump thickness and cell size.</p>



In [ ]:
# Step 5, feature engineering: class balance, standardisation, and removing redundant features.
cat(sprintf("class balance: %d malignant (%.1f%%), %d benign (%.1f%%)\n",
            sum(y == 0), 100*mean(y == 0), sum(y == 1), 100*mean(y == 1)))
scaled <- scale(X)
summ <- data.frame(raw_mean = colMeans(X), raw_sd = apply(X, 2, sd),
                   scaled_mean = round(colMeans(scaled), 10), scaled_sd = apply(scaled, 2, sd))
print(round(summ[c("Cl.thickness", "Cell.size", "Bare.nuclei", "Mitoses"), ], 3))

fc <- abs(cor(X)); drop <- character(0)                   # drop one of every pair with |r| > 0.9
for (i in 1:8) for (j in (i+1):9)
  if (fc[i, j] > 0.9 && !(colnames(X)[i] %in% drop)) drop <- union(drop, colnames(X)[j])
cat("features dropped as redundant (|r| > 0.9 with a kept feature):", length(drop), "->", drop, "\n")



The classes are 35 % malignant and 65 % benign: mild enough to need no correction, but a classifier that always says "benign" is already 65 % accurate, the baseline every reported accuracy must beat. Standardisation gives every feature mean 0 and standard deviation 1, so that features scored on the same 1–10 scale but with different spreads enter distance- and gradient-based methods equally. The redundancy screen drops one feature, cell shape, as nearly a copy of cell size. None of this is "machine learning", and all of it affects the result more than the choice among the classifiers of Section 5.

<div class="box key"><h4>Data science versus machine learning</h4><p>Machine learning is step 6 of seven. A project excellent at step 6 and careless at steps 1, 3 or 7 answers the wrong question, learns the artefacts of its data, or is never used. The methods of this course are necessary, not sufficient.</p></div>

## 3. The paradigms of learning

The paradigms differ in the <em>kind of feedback</em> contained in the experience \(E\), not in the algorithm.



In [ ]:
# Schematic of the three classical learning paradigms (conceptual figure).
set.seed(0)
cen <- rbind(c(-1.5, 0), c(1.5, 0.5))
Xb <- rbind(cbind(rnorm(30, cen[1,1], .8), rnorm(30, cen[1,2], .8)),
            cbind(rnorm(30, cen[2,1], .8), rnorm(30, cen[2,2], .8)))
yb <- rep(0:1, each = 30)
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
# supervised
plot(Xb, col = ifelse(yb == 0, BLUE, ORANGE), pch = 19, cex = .7, xaxt = "n", yaxt = "n",
     xlab = "feature 1", ylab = "feature 2", main = "Supervised: inputs + labels")
abline(a = 0.2, b = 8, lty = 2); legend("bottomleft", c("class 0", "class 1", "boundary"),
     col = c(BLUE, ORANGE, "black"), pch = c(19, 19, NA), lty = c(NA, NA, 2), bty = "n", cex = .8)
# unsupervised
plot(Xb, col = GREY, pch = 19, cex = .7, xaxt = "n", yaxt = "n",
     xlab = "feature 1", ylab = "feature 2", main = "Unsupervised: inputs only")
text(0, 2.6, "find structure: groups, directions, outliers", cex = .8, font = 3)
# reinforcement
plot(0:10, 0:10, type = "n", axes = FALSE, xlab = "", ylab = "", main = "Reinforcement: actions + rewards")
rect(1, 6.2, 9, 8.8, col = adjustcolor(BLUE, .2), border = NA); text(5, 7.5, "Agent (policy)")
rect(1, 1.2, 9, 3.8, col = adjustcolor(ORANGE, .2), border = NA); text(5, 2.5, "Environment\n(patient, game, robot)")
arrows(7.5, 6.1, 7.5, 3.9, length = .12); text(8.3, 5, "action", cex = .8)
arrows(2.5, 3.9, 2.5, 6.1, length = .12); text(1.6, 5, "state,\nreward", cex = .8)
par(op)



<p class="caption"><b>Figure 2.</b> Left: supervised — every input comes with the correct output and the learner fits a function between them. Centre: unsupervised — the same inputs without labels; two groups are visible, and k-means (Section 10) will find them. Right: reinforcement — no dataset at the outset; an agent acts, observes the new state and a scalar reward, and discovers by trial and error which actions lead to high cumulative reward.</p>

### 3.1 Supervised learning

The experience is pairs \((\mathbf{x}_i, y_i)\): input and correct output, supplied by a "supervisor" — a pathologist, a laboratory, a follow-up visit. The task is a function with \(f(\mathbf{x}) \approx y\) on unseen inputs. A categorical \(y\) makes it <strong>classification</strong>; a numerical \(y\) (progression, glucose, survival time) makes it <strong>regression</strong>. They share everything but the loss and the output, so Sections 6–8 do regression and Section 9 turns the machinery into a classifier. Labels encode exactly what we want predicted, which is why most practical impact is supervised; their cost — a radiologist's time, a biopsy, a year of follow-up — is the paradigm's limit.

<div class="box warn"><h4>Terminology: class, label, target</h4><p>In classification <em>class</em> and <em>label</em> are synonyms for the category predicted; <em>target</em> covers both. In regression the target is a continuous value and should <strong>not</strong> be called a class — there are no classes in a regression problem, and the slides' use of "class" for the regression output is a slip. Say <em>target</em> or <em>response</em> for regression, <em>class</em> or <em>label</em> for classification.</p></div>

### 3.2 Two examples from chemistry

Two problems from computational drug discovery — the research field of the laboratory that prepared this material, and the subject of the QSAR unit of BGM324 — show both forms. The instances are molecules and the features are descriptors computed from the structure: molecular weight (MW), hydrogen-bond donors (HBD) and acceptors (HBA), and polar surface area (PSA) — the descriptors of Lipinski's "rule of five" for oral absorption (MW < 500, HBD ≤ 5, HBA ≤ 10; PSA < 140 Å² in later refinements).<sup class="c"><a href="#r6">[6]</a></sup>

<strong>Classification: is the molecule an approved drug?</strong> A binary label — FDA-approved or not — assembled from fda.gov, DrugBank<sup class="c"><a href="#r7">[7]</a></sup> and the Drug Repurposing Hub (clue.io); given the descriptors of a new molecule, predict it.

<strong>Regression: how potent is it?</strong> The <strong>IC50</strong> of a compound against a target is the concentration at which it inhibits the target's activity by 50 %; the lower, the more potent, so compound A with IC50 = 10 µM is a better inhibitor than compound B with 1000 µM. Measured values for millions of compound–target pairs are curated in ChEMBL<sup class="c"><a href="#r8">[8]</a></sup> and BindingDB. Because IC50 spans orders of magnitude, the usual target is <strong>pIC50 = −log<sub>10</sub>(IC50 in mol/L)</strong>: 10 µM is 5, 1 µM is 6, one unit is tenfold in potency, and the errors become closer to Gaussian.

The chunk reproduces both tables with <em>illustrative</em> values (right ranges, not measured compounds). Each follows the workflow of every supervised method: a <em>training step</em>, in which molecules with known labels or pIC50 are given to the algorithm, which adjusts a model's parameters to reproduce them; and a <em>prediction step</em>, in which the descriptors of a new molecule are fed to the fitted model, which outputs a predicted label or pIC50.



In [ ]:
# The two chemistry examples as data frames (illustrative values, not measured data).
drugs <- data.frame(
  MW = c(180.2, 454.6, 285.3, 812.9, 374.4, 1203.3), HBD = c(1, 2, 1, 7, 3, 12),
  HBA = c(3, 6, 4, 13, 5, 19), PSA = c(63.6, 98.1, 46.5, 212.0, 87.3, 345.9),
  FDA_approved = c("yes", "yes", "yes", "no", "yes", "no"),          # the LABEL (class)
  row.names = c("mol_01","mol_02","mol_03","mol_04","mol_05","mol_06"))
potency <- data.frame(
  MW = c(312.4, 298.3, 401.5, 356.8, 275.3), HBD = c(2, 1, 3, 2, 1),
  HBA = c(4, 3, 6, 5, 3), PSA = c(78.2, 55.1, 110.4, 92.0, 48.7),
  IC50_uM = c(10.0, 1000.0, 0.25, 3.2, 150.0),                       # the TARGET (continuous)
  row.names = c("cpd_A","cpd_B","cpd_C","cpd_D","cpd_E"))
potency$pIC50 <- round(-log10(potency$IC50_uM * 1e-6), 2)
cat("CLASSIFICATION: instances x features + a categorical label\n"); print(drugs)
cat("\nREGRESSION: instances x features + a continuous target (pIC50 = -log10 IC50 in mol/L)\n"); print(potency)



Both tables have the shape of the breast-cancer and housing data used below: rows are instances, the last column is what the model must learn — a category in the first, a number in the second. Compound A (pIC50 = 5) beats B (pIC50 = 3) by two orders of magnitude. Real QSAR models use hundreds of descriptors — the situation (\(p\) comparable to \(n\)) in which the regularisation of Section 8 becomes unavoidable.

### 3.3 Unsupervised learning

The experience is the inputs alone, \(\{\mathbf{x}_i\}\). With no correct answer to approximate, the task is reformulated as finding structure, and the slides name four sub-types, each a part of Unit 5. <strong>Clustering</strong> groups instances by similarity (Figure 2, centre; Section 10.1). <strong>Dimensionality reduction</strong> replaces the \(p\) features by a few new ones that retain most of the variation — principal components, or the two-dimensional embeddings used for single-cell data. <strong>Feature selection</strong> keeps a subset of the original features; its motivation is the <strong>curse of dimensionality</strong>: at fixed sample size, adding features first helps a classifier and then hurts it, because the data become sparse in a space of growing volume and the model fits noise in irrelevant features — Hughes showed in 1968 that accuracy peaks at an optimal number of features and then declines.<sup class="c"><a href="#r9">[9]</a></sup> How many features help is answered on held-out data (Section 11); the screen of Section 2.4 is the crudest form. <strong>Association analysis</strong> searches transactional data — purchases, prescriptions, co-occurring diagnoses — for IF–THEN rules ("patients prescribed A and B are also prescribed C") with sufficient support and confidence; the Apriori algorithm is the classical method,<sup class="c"><a href="#r10">[10]</a></sup> and its rules are the data-mining counterpart of expert-system rules, discovered rather than written. Without labels the performance measure is ambiguous — how good is a clustering? — the paradigm's central difficulty, and also what makes it cheap.

### 3.4 Reinforcement learning

The experience is generated by the learner's actions: an <em>agent</em> observes a <em>state</em>, chooses an <em>action</em>, receives a scalar <em>reward</em> and a new state; the task is a <em>policy</em> — states to actions — maximising cumulative reward. The feedback is <em>evaluative</em> (how good the action was, not what the right one was) and <em>delayed</em>, hence the <em>credit-assignment</em> problem of attributing outcomes to earlier actions. Sutton and Barto's textbook is the reference.<sup class="c"><a href="#r11">[11]</a></sup> AlphaGo was reinforcement learning; in medicine it has been used, controversially, to propose fluid and vasopressor dosing in sepsis.<sup class="c"><a href="#r12">[12]</a></sup> Section 10.2 implements the two basic algorithms.

### 3.5 Modern additions: semi-supervised and self-supervised learning

<strong>Semi-supervised learning</strong> uses a small labelled set with a large unlabelled one: the unlabelled data cannot name the classes, but a boundary through a sparse region is more plausible than one cutting a dense cluster.<sup class="c"><a href="#r13">[13]</a></sup> <strong>Self-supervised learning</strong> manufactures labels from the data — hide part of an input (the next word, a masked image patch) and predict it — then <em>fine-tunes</em> the learned representation on a small labelled set; it is how language models are pre-trained and increasingly how medical imaging models are built when labels are scarce.<sup class="c"><a href="#r14">[14]</a></sup>

<div class="box key"><h4>Key idea</h4><p>Paradigms are distinguished by the <em>feedback</em> in \(E\), not by the algorithm. The same network can be trained supervised, self-supervised or by reinforcement. "Uses deep learning" names the model family; it does not yet say what the model learned from.</p></div>

<div class="box bridge"><h4>Bridge to bioinformatics</h4><p>A QSAR model is supervised regression; hierarchical clustering of an expression matrix is unsupervised; protein language models such as ESM are self-supervised, trained to predict masked residues from context.</p></div>

## 4. Approaches to pattern recognition

Unit 1 (Section 3) distinguished symbolic, connectionist, statistical and evolutionary approaches to AI. Pattern recognition inherits the last three; the classical review of the field names the <strong>statistical</strong> and <strong>neural</strong> approaches, and the course plan adds the <strong>evolutionary</strong> one.<sup class="c"><a href="#r15">[15]</a></sup>

| Approach | How the decision rule is obtained | Methods | Units |
|---|---|---|---|
| **Statistical** | Model the classes, or the posterior probability of the class given the features, and estimate parameters from the sample | Bayes classifiers, discriminant analysis, logistic regression, k-NN, trees, SVM | 2–5 |
| **Neural** | Connect simple units with adjustable weights; adjust them iteratively to reduce training error<sup class="c"><a href="#r16">[16]</a></sup> | Perceptron, multi-layer and convolutional networks | 6, 8 |
| **Evolutionary** | Keep a population of candidate rules; score, select, recombine, mutate, repeat<sup class="c"><a href="#r17">[17]</a></sup> | Genetic algorithms, evolution strategies | 7 |

Historically different disciplines, in practice they are three ways of searching for the same thing. The chunk separates malignant from benign aspirates with the two features clump thickness and cell size using one representative of each: linear discriminant analysis computes a line in closed form from class means and a pooled covariance;<sup class="c"><a href="#r38">[38]</a></sup> the perceptron starts from a random line and nudges it after each misclassified case; an evolution strategy keeps a population of 20 random lines, scores them by accuracy and breeds the best five with mutation for 40 generations. R has no perceptron or evolution-strategy function, so both are written from scratch in base R, as the Python edition wrote its own.



In [ ]:
# One problem, three approaches: statistical (LDA), neural (perceptron), evolutionary (ES).
X2 <- as.matrix(bc[, c("Cl.thickness", "Cell.size")]); colnames(X2) <- c("thickness", "size")
Xs <- scale(X2); yv <- y                                     # benign = 1, malignant = 0

lda.fit <- lda(Xs, factor(yv))                               # statistical: closed form
lda.acc <- mean(predict(lda.fit)$class == yv)

tgt <- ifelse(yv == 1, 1, -1); w <- c(0, 0); b <- 0; eta <- 0.1   # neural: perceptron, error-driven
for (ep in 1:50) { errs <- 0
  for (i in sample(nrow(Xs))) { s <- sum(w * Xs[i, ]) + b
    if (tgt[i] * s <= 0) { w <- w + eta * tgt[i] * Xs[i, ]; b <- b + eta * tgt[i]; errs <- errs + 1 } }
  passes <- ep; if (errs == 0) break }
per.acc <- mean(((Xs %*% w + b) > 0) == (yv == 1))

acc_w <- function(wv) mean(((Xs %*% wv[1:2] + wv[3]) > 0) == (yv == 1))   # evolutionary: (5 + 15) ES
set.seed(0); pop <- matrix(rnorm(20 * 3), 20, 3); history <- numeric(40)
for (g in 1:40) { fit <- apply(pop, 1, acc_w); parents <- pop[order(-fit)[1:5], ]; history[g] <- max(fit)
  children <- parents[sample(5, 15, replace = TRUE), ] + matrix(rnorm(15 * 3, 0, 0.3), 15, 3)
  pop <- rbind(parents, children) }
w_es <- pop[which.max(apply(pop, 1, acc_w)), ]
cat(sprintf("training accuracy  LDA %.3f | perceptron %.3f (after %d passes) | ES %.3f (generation 1: %.3f)\n",
            lda.acc, per.acc, passes, acc_w(w_es), history[1]))

op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(Xs, col = ifelse(yv == 0, ORANGE, BLUE), pch = 19, cex = .4,
     xlab = "clump thickness (standardised)", ylab = "cell size (standardised)", main = "Three approaches, one boundary")
xx <- seq(-2, 3, length = 50)
lc <- lda.fit$scaling; lm0 <- (apply(predict(lda.fit)$x, 2, function(.) 0))  # draw LDA via coefficients
abline(-w[1]/w[2]*0 - b/w[2], -w[1]/w[2], lty = 2)                            # perceptron line
abline(-w_es[3]/w_es[2], -w_es[1]/w_es[2], lty = 3, col = GREEN, lwd = 2)     # ES line
legend("topright", c("perceptron (neural)", "ES (evolutionary)"), lty = c(2, 3),
       col = c("black", GREEN), bty = "n", cex = .8)
plot(1:40, history, type = "l", col = GREEN, lwd = 2, xlab = "generation",
     ylab = "best accuracy in population", main = "Evolution strategy: fitness of the best rule")
par(op)



<p class="caption"><b>Figure 3.</b> Left: the 683 aspirates in the standardised feature space and the boundaries found by a neural (dashed) and an evolutionary (dotted green) method. Both are straight lines through the zone of overlap with training accuracies between 0.93 and 0.95. Right: the best accuracy in the evolutionary population rises quickly and then plateaus. LDA needed one computation; the perceptron ran all 50 passes without reaching zero errors, because the classes overlap and no line separates them perfectly; the evolution strategy used 800 accuracy evaluations.</p>

The <em>hypothesis space</em> is the same — straight lines — so none can beat the best line; the approaches differ in <em>how</em> they search and in what else they offer: probabilities (statistical), scale to millions of parameters (neural), optimisation without gradients (evolutionary; Section 7 explains why that matters). Modern practice mixes them freely.

<div class="box key"><h4>Exercise 1</h4><p>Change the mutation scale in the evolution strategy from 0.3 to 0.03 and to 3.0. Small mutations improve slowly; large ones never settle. This is the learning-rate trade-off of Section 7 and the mutation rate of Unit 7.</p></div>

## 5. What a classifier is

### 5.1 Definition and decision boundary

A <strong>classifier</strong> is a function \(\hat{y} = f(\mathbf{x})\) from feature vectors to a finite set of labels \(\{1, \dots, K\}\). Because labels are finite and feature space is continuous, every classifier partitions the space into \(K\) <strong>decision regions</strong> separated by <strong>decision boundaries</strong>; everything it "knows" is where those boundaries lie. Most useful classifiers output a <strong>score</strong> per class — best of all a <strong>probability</strong> \(P(y = k \mid \mathbf{x})\) — and a <em>decision rule</em> turns scores into a label. The separation matters because the decision rule is where the <em>costs</em> of errors enter: a missed malignancy costs far more than an unnecessary biopsy, and "malignant if \(P > 0.5\)" is almost never the right rule (Section 9.5).

### 5.2 The catalogue of the course

The plan lists Bayesian networks, decision trees, k-means, neural networks and genetic algorithms. The chunk trains a representative of four families on the same two features and draws their regions; the fifth, k-means, is treated in Section 10 for the reason in the box below.



In [ ]:
# Four families of classifiers on the same two features: the shapes of their decision regions.
df2 <- data.frame(X2, Class = factor(yv, labels = c("malignant", "benign")))
set.seed(0); n <- nrow(bc); tr <- sample(n, round(0.7 * n))
gx <- seq(0, 11, length = 160); gy <- seq(0, 11, length = 160)
grid <- expand.grid(thickness = gx, size = gy)
models <- list("Gaussian naive Bayes (Unit 3)" = naiveBayes(Class ~ ., df2[tr, ]),
               "k-NN, k = 15 (Unit 3)"        = NULL,
               "Decision tree, depth 3 (Unit 4)" = rpart(Class ~ ., df2[tr, ], control = rpart.control(maxdepth = 3)),
               "Neural net, 8 units (Unit 6)" = nnet(Class ~ ., df2[tr, ], size = 8, maxit = 500, trace = FALSE))
op <- par(mfrow = c(1, 4), mar = c(4, 3, 2.4, 1))
accs <- numeric(4); nm <- names(models)
for (k in 1:4) {
  if (k == 2) { pr <- knn(scale(X2)[tr, ], as.matrix(scale(X2))[-tr, ], df2$Class[tr], k = 15)
    gpred <- knn(scale(X2)[tr, ], scale(grid, center = attr(scale(X2),"scaled:center"), scale = attr(scale(X2),"scaled:scale")), df2$Class[tr], k = 15)
  } else { m <- models[[k]]
    pr <- if (k == 3) predict(m, df2[-tr, ], type = "class") else predict(m, df2[-tr, ], type = "class")
    gpred <- if (k == 3) predict(m, grid, type = "class") else predict(m, grid, type = "class") }
  accs[k] <- mean(pr == df2$Class[-tr])
  image(gx, gy, matrix(as.integer(gpred == "benign"), 160), col = c("#fbe3d3", "#dfe7f1"),
        xlab = "clump thickness", ylab = if (k == 1) "cell size" else "", main = sprintf("%s\nacc %.3f", nm[k], accs[k]), cex.main = .8)
  points(X2[-tr, ], col = ifelse(df2$Class[-tr] == "benign", BLUE, ORANGE), pch = 19, cex = .35)
}
par(op)



<p class="caption"><b>Figure 4.</b> Decision regions (orange malignant, blue benign) of four families trained on the same 478 cases and evaluated on the same 205 held-out cases (dots). Naive Bayes gives smooth curves from Gaussians per class; k-nearest neighbours an irregular boundary following the local majority; the tree axis-parallel rectangles, each decision comparing one feature with one threshold; the network a smooth curve of free shape. Test accuracies lie between 0.93 and 0.94 — no family is decisively better, a finding documented across many benchmarks.<sup class="c"><a href="#r18">[18]</a></sup> The differences that matter are interpretability, cost, data requirements and robustness.</p>

<div class="box warn"><h4>A correction to the plan: k-means is not a classifier</h4><p>The plan lists k-means among "classification techniques". <strong>k-means is a clustering algorithm</strong>: it partitions unlabelled data into \(k\) groups by minimising within-group variance and never sees or predicts a class label.<sup class="c"><a href="#r19">[19]</a></sup><sup class="c"><a href="#r20">[20]</a></sup> Its output <em>looks</em> like a classification and it shares a letter with <em>k-nearest neighbours</em>, but the two have nothing else in common. Accepted statement: k-means is the canonical unsupervised method (Unit 5, Section 10.1); k-nearest neighbours is a supervised classifier (Unit 3, Figure 4). Likewise "Bayesian networks" are a general formalism of which the Bayesian <em>classifiers</em> of Unit 3 are the special case the course develops, and "genetic algorithms" are an optimisation method (Section 4, Unit 7) that can build classifiers but is not one.</p></div>

<div class="box key"><h4>What the rest of this unit does</h4><p>Figure 4 shows what classifiers <em>produce</em>; nothing so far shows how their parameters are <em>found</em>, which is what learning means. Sections 6–9 answer that for the simplest models — a straight line for a number, a straight line for a probability — in complete detail. Every later unit is a variation on the same three steps.</p></div>

## 6. Linear regression: the first learning problem

For the regression sections the Python edition used a diabetes-progression dataset;<sup class="c"><a href="#r21">[21]</a></sup> the R edition uses the <strong>Boston housing</strong> benchmark shipped with <code>mlbench</code><sup class="c"><a href="#r37">[37]</a></sup> — the canonical teaching dataset for regression in R and the running example of *Introduction to Statistical Learning*<sup class="c"><a href="#r39">[39]</a></sup> — because it shows the whole arc of regression cleanly (a single feature already explains a third of the variance, and the full model much more). It records 506 census tracts of 1970s Boston, thirteen variables per tract — crime rate, average number of rooms (<code>rm</code>), proportion of lower-status population (<code>lstat</code>), pupil–teacher ratio, distance to employment centres and others — and the median owner-occupied home value in thousands of dollars (<code>medv</code>) as the target. The mechanics are identical to any clinical regression; Section 9 returns to medical data.



In [ ]:
# The Boston housing dataset: 506 tracts x 13 variables, and the target medv (median value, $1000s).
data(BostonHousing, package = "mlbench")
bh <- BostonHousing; bh$chas <- as.numeric(as.character(bh$chas))
yb <- bh$medv; Xd <- as.matrix(bh[, setdiff(names(bh), "medv")])
cat(sprintf("X: %d x %d | target medv: mean %.1f, sd %.1f, range %d-%d\n",
            nrow(Xd), ncol(Xd), mean(yb), sd(yb), min(yb), max(yb)))
print(round(t(apply(Xd[, c("rm","lstat","crim","tax","ptratio")], 2,
                    function(z) c(mean = mean(z), sd = sd(z), min = min(z), max = max(z)))), 2))



Thirteen variables on very different scales — rooms around 6, tax in the hundreds — which matters in Section 7. The target has mean 22.5 and standard deviation 9.2: a predictor that always outputs the mean errs by about 9.2 (in $1000s) typically, and any model must do better.

### 6.1 The model

<strong>Regression</strong> is supervised learning with a numerical target. The simplest model makes the target a linear function of the features plus noise:

\[ \hat{y} = f_{\boldsymbol\beta}(\mathbf{x}) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p = \mathbf{x}^\top \boldsymbol\beta \]

where the second form absorbs the intercept by prepending a constant feature \(x_0 = 1\), so that \(\mathbf{x}\) and \(\boldsymbol\beta\) have length \(p + 1\). The hypothesis space is the set of hyperplanes, the parameters the \(p + 1\) coefficients. The name is Galton's 1886 "regression towards mediocrity" of children's heights, which stuck to the method he used — fitting a line.<sup class="c"><a href="#r22">[22]</a></sup>

### 6.2 The loss: least squares

The classical measure of fit is the <strong>sum of squared residuals</strong>, \(r_i = y_i - \hat{y}_i\) being the vertical distance from observation to line:

\[ L(\boldsymbol\beta) = \sum_{i=1}^{n} \left(y_i - \mathbf{x}_i^\top \boldsymbol\beta\right)^2 = \lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2 \]

where \(\mathbf{X}\) is the \(n \times (p+1)\) data matrix with a leading column of ones. Divided by \(n\) it is the <em>mean squared error</em> (MSE), with the same minimiser; its square root (RMSE) is in the units of \(y\). Squares are differentiable, so the minimum has a closed form; they penalise a few large errors more than many small ones; and under Gaussian noise minimising squared error is maximising the likelihood of the data — the principle behind every loss in this course, used in Section 9 for logistic regression. Legendre published the method in 1805, Gauss claimed it from 1795, and the dispute is one of the best documented in the history of science.<sup class="c"><a href="#r23">[23]</a></sup>

### 6.3 Worked example by hand: four points

Four observations with a single feature \(x\) and target \(y\) (invented round values):

| observation | \(x\) | \(y\) |
|---|---|---|
| 1 | 20 | 100 |
| 2 | 24 | 120 |
| 3 | 28 | 180 |
| 4 | 32 | 200 |

For one feature the least-squares line \(\hat{y} = \beta_0 + \beta_1 x\) has the textbook solution:

\[ \beta_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}, \qquad \beta_0 = \bar{y} - \beta_1 \bar{x}. \]

**Step 1 — means.** \(\bar{x} = (20 + 24 + 28 + 32)/4 = 26\); \(\bar{y} = (100 + 120 + 180 + 200)/4 = 150\).

**Step 2 — deviations.**

| \(i\) | \(x_i - \bar{x}\) | \(y_i - \bar{y}\) | product | \((x_i - \bar{x})^2\) |
|---|---|---|---|---|
| 1 | −6 | −50 | 300 | 36 |
| 2 | −2 | −30 | 60 | 4 |
| 3 | 2 | 30 | 60 | 4 |
| 4 | 6 | 50 | 300 | 36 |
| **sum** | | | **720** | **80** |

**Step 3 — coefficients.** \(\beta_1 = 720/80 = 9\) units of \(y\) per unit of \(x\); \(\beta_0 = 150 - 9 \times 26 = -84\).

**Step 4 — check.** Predictions \((96, 132, 168, 204)\); residuals \((4, -12, 12, -4)\), summing to zero as least-squares residuals always do with an intercept; squared sum \(320\), MSE \(80\). The chunk confirms the arithmetic and that no other slope does better.



In [ ]:
# The four-point example: the hand-computed line, and that nearby slopes have a larger loss.
x4 <- c(20, 24, 28, 32); y4 <- c(100, 120, 180, 200)
b1 <- sum((x4 - mean(x4)) * (y4 - mean(y4))) / sum((x4 - mean(x4))^2)
b0 <- mean(y4) - b1 * mean(x4); resid4 <- y4 - (b0 + b1 * x4)
cat(sprintf("slope = %.1f, intercept = %.1f, residuals = %s, sum = %.1f, MSE = %.1f\n",
            b1, b0, paste(resid4, collapse = ", "), sum(resid4), mean(resid4^2)))
mse <- function(a, s) mean((y4 - (a + s * x4))^2)
cat(sprintf("MSE of hand line: %.1f | MSE with slope 8: %.1f | with slope 10: %.1f\n",
            mse(b0, b1), mse(mean(y4) - 8 * mean(x4), 8), mse(mean(y4) - 10 * mean(x4), 10)))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(x4, y4, pch = 19, col = BLUE, cex = 1.4, xlab = "feature x", ylab = "target y", main = "The fitted line and its residuals")
abline(b0, b1, col = ORANGE, lwd = 2); segments(x4, y4, x4, b0 + b1 * x4, col = GREY)
legend("topleft", sprintf("y = %.0f + %.0f x", b0, b1), lty = 1, col = ORANGE, bty = "n", cex = .9)
slopes <- seq(4, 14, length = 100)
plot(slopes, sapply(slopes, function(s) mse(mean(y4) - s * mean(x4), s)), type = "l", col = BLUE, lwd = 2,
     xlab = "slope (intercept optimal for each slope)", ylab = "mean squared error", main = "The loss is a parabola; minimum at 9")
points(b1, mse(b0, b1), pch = 19, col = ORANGE, cex = 1.5)
par(op)



<p class="caption"><b>Figure 5.</b> Left: the four observations, the line \(\hat{y} = -84 + 9x\) and the residuals (grey), whose squares the method minimises. Right: the MSE as a function of the slope, with the intercept re-optimised for each slope — a parabola with a unique minimum at slope 9 and MSE 80. With one minimum, any downhill method finds it; logistic regression shares this property.</p>

### 6.4 The normal equations

Expanding the loss in matrix form and setting the gradient to zero:

\[ \nabla_{\boldsymbol\beta} L = -2\,\mathbf{X}^\top \mathbf{y} + 2\,\mathbf{X}^\top \mathbf{X}\,\boldsymbol\beta = \mathbf{0} \;\Longrightarrow\; \mathbf{X}^\top \mathbf{X}\, \hat{\boldsymbol\beta} = \mathbf{X}^\top \mathbf{y} \;\Longrightarrow\; \hat{\boldsymbol\beta} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}. \]

These are the <strong>normal equations</strong>: \(p + 1\) linear equations in \(p + 1\) unknowns, uniquely solvable when no feature is an exact linear combination of the others and \(n > p\). The name is geometric: the residual vector is <em>normal</em> to every column of \(\mathbf{X}\), so the fitted values are the projection of \(\mathbf{y}\) onto the feature space. In code one never forms the inverse: R's <code>solve</code> solves the system directly, and <code>lm</code> uses a QR decomposition.



In [ ]:
# Normal equations with one feature (rm = average rooms), checked against lm().
xr <- bh$rm; X1 <- cbind(1, xr)
beta1 <- solve(t(X1) %*% X1, t(X1) %*% yb)                   # solve (X'X) beta = X'y
yhat1 <- X1 %*% beta1; rss <- sum((yb - yhat1)^2); tss <- sum((yb - mean(yb))^2)
cat("X'X =\n"); print(round(t(X1) %*% X1, 0))
cat(sprintf("beta (intercept, slope) = (%.3f, %.3f) | RMSE = %.2f | R^2 = %.3f\n",
            beta1[1], beta1[2], sqrt(rss / length(yb)), 1 - rss / tss))
print(round(coef(lm(yb ~ xr)), 3))                           # lm() agrees exactly
plot(xr, yb, pch = 19, col = adjustcolor(BLUE, .4), cex = .5, xlab = "average rooms per dwelling",
     ylab = "median home value ($1000s)", main = "Least-squares line: value against rooms")
abline(beta1[1], beta1[2], col = ORANGE, lwd = 2)
legend("topleft", sprintf("y = %.0f + %.1f x", beta1[1], beta1[2]), lty = 1, col = ORANGE, bty = "n")



<p class="caption"><b>Figure 6.</b> Median value against average rooms for 506 tracts and the least-squares line, slope about 9.1: each additional room is associated with about $9,100 more in median value on average. The scatter is wide — RMSE about 6.6 against the 9.2 of the do-nothing predictor — so rooms alone explain a modest but real fraction of the variation (\(R^2 = 0.48\)).</p>

### 6.5 Reading the output: coefficients and R²

A <strong>coefficient</strong> says how the prediction changes with its feature, <em>all else held equal</em> — vacuous with one feature, essential with thirteen. The <strong>coefficient of determination</strong>

\[ R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2} = 1 - \frac{\text{RSS}}{\text{TSS}} \]

compares the model's residual sum of squares with that of the do-nothing predictor \(\hat{y} = \bar{y}\): 1 means every point on the line, 0 no better than the mean; on training data it cannot be negative when an intercept is fitted, on new data it can.



In [ ]:
# Normal equations with all thirteen features. The code is the same; only the design matrix is wider.
X13 <- cbind(1, Xd); beta13 <- solve(t(X13) %*% X13, t(X13) %*% yb); yhat13 <- X13 %*% beta13
r2_13 <- 1 - sum((yb - yhat13)^2) / tss
cat(sprintf("R^2 with 13 features = %.3f (rooms alone: %.3f) | RMSE = %.2f\n",
            r2_13, 1 - rss / tss, sqrt(mean((yb - yhat13)^2))))
coef <- data.frame(per_unit = beta13[-1], feature_sd = apply(Xd, 2, sd))
coef$effect_of_1sd <- coef$per_unit * coef$feature_sd
print(round(coef[order(-abs(coef$effect_of_1sd)), ], 3))



\(R^2\) rises to 0.74 and the RMSE falls to 4.7. The rooms coefficient drops from 9.1 to 3.8: part of what rooms "explained" alone is now attributed to variables it correlates with. Raw coefficients are not comparable, so the third column rescales each per standard deviation of its feature: lower-status proportion (<code>lstat</code>), distance (<code>dis</code>) and nitric-oxide concentration contribute most negatively, rooms and access to radial highways (<code>rad</code>) most positively. This reading of coefficients is the basis of every multivariable risk model in medicine.

<div class="box warn"><h4>Two cautions</h4><p>A coefficient describes how the <em>prediction</em> changes, not what would happen if the feature changed. And when features are strongly correlated — here property tax and highway access (<code>tax</code>, <code>rad</code>) correlate at 0.91 — \(\mathbf{X}^\top\mathbf{X}\) is nearly singular and coefficients become unstable: large, of opposite signs, sensitive to a few cases, while predictions stay fine. The pair <code>tax</code> (−2.1 per standard deviation) and <code>rad</code> (+2.7) is this instability. Ridge regression (Section 8.3) was invented for it.<sup class="c"><a href="#r24">[24]</a></sup></p></div>

### 6.6 Residual plots

\(R^2\) hides <em>where</em> the model fails; residuals show it. Against fitted values they should form a structureless band — a curve means non-linearity, a funnel noise growing with the prediction; their histogram should be roughly symmetric.



In [ ]:
# Residual diagnostics for the thirteen-feature model.
resid13 <- yb - yhat13
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(yhat13, resid13, pch = 19, col = adjustcolor(BLUE, .4), cex = .5, xlab = "fitted value",
     ylab = "residual (observed - fitted)", main = "Residuals against fitted values"); abline(h = 0)
hist(resid13, breaks = 30, col = adjustcolor(BLUE, .7), xlab = "residual", main = "Distribution of residuals"); abline(v = 0)
par(op)
skew <- mean((resid13 - mean(resid13))^3) / sd(resid13)^3
cat(sprintf("residual mean = %.2e (zero by construction) | sd = %.2f | skewness = %.2f | largest |residual| = %.1f\n",
            mean(resid13), sd(resid13), skew, max(abs(resid13))))



<p class="caption"><b>Figure 7.</b> Left: residuals against fitted values. A diagonal streak of large positive residuals appears at the right, where the model systematically under-predicts the most expensive tracts: median value was <em>capped at 50</em> in the survey (a ceiling effect a linear model cannot represent). Right: residuals are right-skewed (skewness 1.52), standard deviation 4.7, largest residual 26. The skew is the signature of that ceiling; the next step would be a transformed target or a non-linear model (Unit 4). The Python edition saw the mirror-image floor effect in its data — the same methodological lesson.</p>

<div class="box key"><h4>Exercise 2</h4><p>(a) Verify on paper that for one feature with an intercept the normal equations are \(n\beta_0 + \beta_1 \sum x_i = \sum y_i\) and \(\beta_0 \sum x_i + \beta_1 \sum x_i^2 = \sum x_i y_i\), and that solving them gives the formulas of Section 6.3. (b) Remove the columns <code>tax</code> and <code>rad</code> from the thirteen-feature chunk and re-fit. How much do \(R^2\) and the remaining coefficients change?</p></div>

## 7. Gradient descent: how a machine learns

### 7.1 Why a closed form is not enough

Almost no learning problem has a closed form: logistic regression (Section 9) has none, a neural network's loss surface has no formula for its minimum, and even linear regression with \(p\) in the hundreds of thousands cannot form \(\mathbf{X}^\top\mathbf{X}\). What every problem has is a loss that can be <em>evaluated</em> and <em>differentiated</em>. <strong>Gradient descent</strong> uses nothing else, trains essentially every modern model, and is the moment at which "the machine learns" stops being a metaphor.

### 7.2 The idea

The gradient \(\nabla L(\theta)\) is the vector of partial derivatives; it points uphill, so its negative points downhill. Start anywhere and repeat

\[ \theta^{(t+1)} = \theta^{(t)} - \eta\, \nabla L(\theta^{(t)}) \]

where \(\eta > 0\) is the <strong>learning rate</strong>; stop when the gradient is nearly zero, the loss stops changing, or after a fixed number of steps. The learning rate has three regimes: too small, correct but slow; too large, each step overshoots and the loss <em>diverges</em>; in between, fast convergence. For a quadratic loss the boundary is exact — \(\eta < 2/\lambda_{\max}\), the largest eigenvalue of the Hessian — and the speed is set by the ratio of largest to smallest eigenvalue, the <em>condition number</em>, large when features have different scales. Hence feature scaling, optional for the normal equations, is almost mandatory for gradient descent.

### 7.3 The gradient of the least-squares loss

For the MSE \(L(\boldsymbol\beta) = \frac{1}{n}\lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2\),

\[ \nabla_{\boldsymbol\beta} L = -\frac{2}{n}\, \mathbf{X}^\top (\mathbf{y} - \mathbf{X}\boldsymbol\beta) = -\frac{2}{n}\, \mathbf{X}^\top \mathbf{r}, \]

the correlation of each feature with the residuals: if a feature is high where the model under-predicts, its coefficient goes up. The chunk implements the three-line update for one feature — <code>lstat</code>, chosen because its large spread makes the loss surface ill-conditioned and the three regimes vivid — <em>centred</em> (so the intercept is the mean of \(y\)) at three learning rates.



In [ ]:
# Batch gradient descent for least squares, by hand. Three learning rates: too small, good, too large.
xl <- bh$lstat; xc <- xl - mean(xl); Xc <- cbind(1, xc)       # centred feature
beta_closed <- solve(t(Xc) %*% Xc, t(Xc) %*% yb)              # closed form for comparison
gradient_descent <- function(Xm, yv, lr, steps = 100) {
  beta <- c(0, 0); path <- matrix(0, steps + 1, 2); losses <- numeric(steps); path[1, ] <- beta
  for (t in 1:steps) { r <- yv - Xm %*% beta; losses[t] <- mean(r^2)
    beta <- beta - lr * (-2 / length(yv) * t(Xm) %*% r); path[t + 1, ] <- beta }
  list(path = path, losses = losses) }
lam_max <- 2 * max(eigen(t(Xc) %*% Xc / length(yb))$values)
cat(sprintf("largest Hessian eigenvalue = %.1f -> converges only if lr < 2/%.1f = %.4f\n", lam_max, lam_max, 2 / lam_max))
runs <- lapply(c(0.002, 0.015, 0.021), function(lr) gradient_descent(Xc, yb, lr))
for (k in 1:3) { lr <- c(0.002, 0.015, 0.021)[k]
  cat(sprintf("lr = %-6g after 100 steps: beta = (%.2f, %.2f)  MSE = %.4g\n",
              lr, runs[[k]]$path[101, 1], runs[[k]]$path[101, 2], runs[[k]]$losses[100])) }
cat(sprintf("closed form:            beta = (%.2f, %.2f)  MSE = %.4g\n",
            beta_closed[1], beta_closed[2], mean((yb - Xc %*% beta_closed)^2)))



The three regimes appear. At \(\eta = 0.002\) the loss is still falling after 100 steps and the intercept (7.4) is far from 22.5, while the slope — the direction of larger curvature — converged long ago. At \(\eta = 0.015\) both parameters match the closed form and the MSE equals the minimum, about 38.5. At \(\eta = 0.021\), just above the threshold of 0.0196, the slope oscillates with growing amplitude and the loss reaches order \(10^{12}\).



In [ ]:
# Convergence curves for the three learning rates, and the path of two on the loss surface.
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
cols <- c(GREY, BLUE, ORANGE); lrs <- c(0.002, 0.015, 0.021)
plot(NA, xlim = c(1, 100), ylim = c(1, 1e6), log = "y", xlab = "step",
     ylab = "mean squared error (log, clipped)", main = "Loss per step for three learning rates")
for (k in 1:3) lines(pmin(pmax(runs[[k]]$losses, 1), 1e6), col = cols[k], lwd = 1.5)
abline(h = mean((yb - Xc %*% beta_closed)^2), lty = 2)
legend("topright", c("lr = 0.002", "lr = 0.015", "lr = 0.021 (diverges)", "closed-form min"),
       col = c(cols, "black"), lty = c(1, 1, 1, 2), bty = "n", cex = .8)
b0g <- seq(0, 45, length = 120); b1g <- seq(-3, 1, length = 120)
L <- outer(b0g, b1g, Vectorize(function(a, s) mean((yb - (a + s * xc))^2)))
contour(b0g, b1g, L, nlevels = 14, col = GREY, xlab = "intercept", ylab = "slope",
        main = "Descent on the loss surface (MSE contours)")
for (k in 1:2) { p <- runs[[k]]$path; lines(p[, 1], p[, 2], col = cols[k], type = "o", pch = 19, cex = .3) }
points(beta_closed[1], beta_closed[2], pch = 8, col = ORANGE, cex = 2)
par(op)



<p class="caption"><b>Figure 8.</b> Left: MSE per step for the three rates (log axis). The small rate descends steadily but has not reached the minimum (dashed) after 100 steps; the moderate rate reaches it in a few dozen; the large rate decreases briefly and then grows without bound (clipped at \(10^6\)). Right: the loss surface over (intercept, slope). The slope direction is curved about fifty times more sharply than the intercept direction (the two axes are drawn on very different scales, so the genuinely elongated contours appear round here); the moderate-rate path descends the steep slope direction in a few oscillating steps and then creeps along the flat intercept direction. This is the condition number made visible; standardising the feature would equalise the curvatures.</p>

<div class="box key"><h4>This is what learning is</h4><p>Nothing in the gradient-descent chunk knows about housing or lines; it knows a loss and its gradient and walks downhill. Replace the model by a network with a million weights and the loss by that of Section 9, keep the three lines of the update, and you have the training loop of every deep-learning system.</p></div>

### 7.4 Stochastic gradient descent

The gradient above used all \(n\) cases at every step — <em>batch</em> descent. With a million images that is unnecessary: a gradient from a random <em>mini-batch</em> is a noisy but unbiased estimate of the full one. <strong>Stochastic gradient descent</strong> (SGD) does this with a decreasing learning rate so the noise averages out; the convergence proof is Robbins and Monro's 1951 stochastic approximation.<sup class="c"><a href="#r25">[25]</a></sup> The chunk implements mini-batch SGD from scratch (R has no built-in SGD regressor) on all thirteen features, with and without scaling.



In [ ]:
# Mini-batch stochastic gradient descent from scratch, on raw vs standardised features.
sgd <- function(Xm, yv, lr0, epochs = 60, bs = 32) {
  Xa <- cbind(1, Xm); beta <- rep(0, ncol(Xa)); nn <- nrow(Xa); it <- 0
  for (e in 1:epochs) { idx <- sample(nn)
    for (s in seq(1, nn, bs)) { bi <- idx[s:min(s + bs - 1, nn)]; it <- it + 1
      r <- yv[bi] - Xa[bi, ] %*% beta; lr <- lr0 / (1 + it * 1e-3)
      beta <- beta - lr * (-2 / length(bi) * t(Xa[bi, ]) %*% r) } }
  1 - sum((yv - Xa %*% beta)^2) / sum((yv - mean(yv))^2) }
r2_ols <- summary(lm(yb ~ Xd))$r.squared
set.seed(0); r2_raw <- sgd(Xd, yb, 1e-5); set.seed(0); r2_scaled <- sgd(scale(Xd), yb, 0.05)
res <- data.frame(method = c("normal equations", "SGD on raw features", "SGD on standardised features"),
  R2_training = c(sprintf("%.3f", r2_ols),
                  if (is.finite(r2_raw)) sprintf("%.3f", r2_raw) else "diverged",
                  sprintf("%.3f", r2_scaled)))
print(res, row.names = FALSE)



On standardised features SGD reaches the \(R^2\) of the normal equations (0.74 against 0.74) in a few dozen passes. On raw features — tax in the hundreds, crime near zero — it diverges: no single rate suits all directions of so elongated a surface. Standardise before any gradient-based method.

<div class="box key"><h4>Exercise 3</h4><p>Replace the centred feature <code>xc</code> by the standardised one <code>(xl - mean(xl)) / sd(xl)</code> and re-run both chunks: the threshold becomes <code>lr &lt; 1</code> and the contours circles. Find a rate that reaches the minimum in fewer than ten steps. Then run the batch descent with <code>steps = 2000</code> and <code>lr = 0.002</code> on the centred feature and confirm that the small rate gets there in the end.</p></div>

## 8. Overfitting: a preview

### 8.1 More flexible is not better

The linear model is often wrong: dose–response curves saturate, signals oscillate. <strong>Polynomial regression</strong> replaces \(x\) by \(x, x^2, \dots, x^d\) and fits a linear model to them — linear in the <em>parameters</em>, so the normal equations apply unchanged, but the curve can bend, and the degree \(d\) controls flexibility. The experiment draws 20 training points from a sine curve plus noise, fits degrees 1, 3 and 15, and evaluates each on the training points <em>and</em> on 200 new points from the same curve.



In [ ]:
# Polynomial regression on a synthetic 1-D problem: training error versus error on new data.
truth <- function(x) sin(2 * pi * x)
set.seed(0); xtr <- sort(runif(20)); ytr <- truth(xtr) + rnorm(20, 0, 0.3)
set.seed(1); xte <- sort(runif(200, min(xtr), max(xtr))); yte <- truth(xte) + rnorm(200, 0, 0.3)
xx <- seq(min(xtr), max(xtr), length = 400)
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1)); rows <- list()
for (k in seq_along(c(1, 3, 15))) { d <- c(1, 3, 15)[k]
  m <- lm(ytr ~ poly(xtr, d))
  tr_mse <- mean((predict(m) - ytr)^2); te_mse <- mean((predict(m, data.frame(xtr = xte)) - yte)^2)
  rows[[k]] <- data.frame(degree = d, parameters = d + 1, train_MSE = tr_mse, test_MSE = te_mse)
  plot(xtr, ytr, pch = 19, col = BLUE, cex = .8, ylim = c(-2, 2), xlab = "x", ylab = if (k==1) "y" else "",
       main = sprintf("degree %d: train %.3f, test %.3f", d, tr_mse, te_mse), cex.main = .85)
  lines(xx, truth(xx), col = GREY, lty = 2); lines(xx, predict(m, data.frame(xtr = xx)), col = ORANGE, lwd = 2) }
par(op); print(do.call(rbind, rows), row.names = FALSE)



<p class="caption"><b>Figure 9.</b> Three polynomial fits (orange) to 20 noisy points (blue) from a sine curve (dashed). Degree 1 cannot follow the curve and errs on both sets: it <strong>underfits</strong>. Degree 3 follows the curve with the lowest test error, about 0.10 — essentially the noise variance \(0.3^2 = 0.09\), the best any model could do. Degree 15 nearly passes through the training points (training MSE 0.008) but swings wildly between them; its test error is enormous: it <strong>overfits</strong>, having learned the noise of the sample rather than the pattern.</p>

### 8.2 Training error is not the quantity of interest

The table states the most important methodological lesson of the course: <strong>training error decreases monotonically with flexibility and is useless for choosing the model</strong>. What matters is the <em>generalisation</em> error on unseen data, which falls, reaches a minimum at the right flexibility and rises again. Supplement S1 develops the methodology; here the point is to recognise the phenomenon, because every classifier in Figure 4 has a flexibility knob — tree depth, neighbours, network size — and every one can overfit.

### 8.3 Regularisation: ridge regression

Degree 15 is under-constrained: nothing stops the coefficients growing to absurd values to thread the noise. <strong>Regularisation</strong> adds a penalty on coefficient size to the loss, trading fit against simplicity. <strong>Ridge regression</strong> (Hoerl and Kennard, 1970)<sup class="c"><a href="#r24">[24]</a></sup> is the oldest form:

\[ L_{\text{ridge}}(\boldsymbol\beta) = \lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2 + \lambda \lVert \boldsymbol\beta \rVert^2 \;\Longrightarrow\; \hat{\boldsymbol\beta}_{\text{ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y} \]

where \(\lambda \ge 0\) sets the strength. The solution is the normal equations with \(\lambda\) on the diagonal — the "ridge" — which makes the matrix invertible even when \(\mathbf{X}^\top\mathbf{X}\) is singular and shrinks all coefficients towards zero. The chunk applies this formula to the degree-15 features, standardised so the penalty treats all powers equally.



In [ ]:
# Ridge regression on the degree-15 features: the normal equations with lambda on the diagonal.
poly_design <- function(xv, d, lo, hi) { t <- 2 * (xv - lo) / (hi - lo) - 1; sapply(1:d, function(k) t^k) }
lo <- min(xtr); hi <- max(xtr); Phi <- poly_design(xtr, 15, lo, hi)
mu <- colMeans(Phi); sdv <- apply(Phi, 2, sd)
Ptr <- scale(Phi); Pte <- scale(poly_design(xte, 15, lo, hi), mu, sdv); Pxx <- scale(poly_design(xx, 15, lo, hi), mu, sdv)
plot(xtr, ytr, pch = 19, col = BLUE, cex = .8, ylim = c(-2, 2), xlab = "x", ylab = "y",
     main = "Ridge tames the degree-15 polynomial"); lines(xx, truth(xx), col = GREY, lty = 2)
rows <- list(); lcol <- c(ORANGE, GREEN, "black"); k <- 0
for (lam in c(0, 1e-4, 1e-2, 1, 100)) {
  b <- solve(t(Ptr) %*% Ptr + lam * diag(15), t(Ptr) %*% (ytr - mean(ytr)))
  pred <- function(P) P %*% b + mean(ytr)
  rows[[length(rows)+1]] <- data.frame(lambda = lam, train_MSE = mean((pred(Ptr) - ytr)^2),
    test_MSE = mean((pred(Pte) - yte)^2), coef_norm = sqrt(sum(b^2)))
  if (lam %in% c(0, 1e-2, 100)) { k <- k + 1; lines(xx, pred(Pxx), col = lcol[k], lwd = 2) } }
legend("topright", c("lambda = 0", "lambda = 0.01", "lambda = 100"), col = lcol, lty = 1, lwd = 2, bty = "n", cex = .8)
print(do.call(rbind, rows), row.names = FALSE)



<p class="caption"><b>Figure 10.</b> The degree-15 polynomial under three penalties. \(\lambda = 0\) (orange) is the overfit of Figure 9, coefficient norm about 11,000. \(\lambda = 0.01\) (green): the same sixteen parameters give a smooth curve tracking the truth as well as the cubic — test MSE about 0.12, coefficient norm 2.8, four orders of magnitude smaller. \(\lambda = 100\) (black): the penalty dominates and the curve underfits. Regularisation strength is a second flexibility knob, chosen like the degree on held-out data (Supplement S1).</p>

<div class="box bridge"><h4>Bridge to later units and to QSAR</h4><p>Every model has such a knob: <code>C</code> in logistic regression and support vector machines (Unit 3) is an inverse \(\lambda\); leaf size of a tree (Unit 4) and weight decay of a network (Unit 6) play the same role. In QSAR (BGM324), with hundreds of descriptors and a few dozen molecules, \(p > n\) and ridge and its relatives (lasso, elastic net, partial least squares) are the only way to fit a linear model at all. Theory: Hastie, Tibshirani and Friedman, chapter 3.<sup class="c"><a href="#r26">[26]</a></sup></p></div>

## 9. Logistic regression: the first classifier

### 9.1 From a linear score to a probability

The simplest classifier reuses the linear model: a score \(z = \mathbf{x}^\top\boldsymbol\beta\), benign if \(z > 0\), as in Figure 3. But a score is not a probability, and fitting it by least squares to labels 0 and 1 predicts 1.4 or −0.2 and is pulled around by cases already far on the correct side. <strong>Logistic regression</strong> passes the score through the <strong>sigmoid</strong>:

\[ p(\mathbf{x}) = P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{x}^\top\boldsymbol\beta) = \frac{1}{1 + e^{-\mathbf{x}^\top\boldsymbol\beta}} \]

with \(\sigma(0) = 0.5\), \(\sigma(z) \to 1\) as \(z \to \infty\), and \(\sigma(-z) = 1 - \sigma(z)\). Inverting: \(\mathbf{x}^\top\boldsymbol\beta = \log\frac{p}{1-p}\) — the score is the <strong>log-odds</strong> (logit), each coefficient is a change in log-odds per unit of its feature, and \(e^{\beta_j}\) is an <em>odds ratio</em>, the quantity reported in every clinical paper that uses the model. The boundary at threshold 0.5 is still the hyperplane \(\mathbf{x}^\top\boldsymbol\beta = 0\); the sigmoid adds a calibrated statement of <em>how far</em> from it a case is. The regression formulation with maximum-likelihood fitting is Cox's (1958).<sup class="c"><a href="#r27">[27]</a></sup>



In [ ]:
# The sigmoid function and its derivative.
sigmoid <- function(z) 1 / (1 + exp(-z)); z <- seq(-8, 8, length = 400)
plot(z, sigmoid(z), type = "l", col = BLUE, lwd = 2, xlab = "linear score z (log-odds)",
     ylab = "probability", main = "The sigmoid maps any score to a probability")
lines(z, sigmoid(z) * (1 - sigmoid(z)), col = ORANGE, lwd = 1.5, lty = 2)
abline(h = 0.5, col = GREY); abline(v = 0, col = GREY)
for (zz in c(-4, -2, 0, 2, 4)) { points(zz, sigmoid(zz), pch = 19, col = BLUE)
  text(zz + 0.6, sigmoid(zz) - 0.04, sprintf("%.3f", sigmoid(zz)), cex = .75) }
legend("left", c("sigmoid", "derivative s(z)(1-s(z))"), col = c(BLUE, ORANGE), lty = c(1, 2), bty = "n", cex = .8)



<p class="caption"><b>Figure 11.</b> The sigmoid and its derivative. A score of 0 is probability 0.5; ±2 gives 0.88 and 0.12; ±4 gives 0.98 and 0.02. The derivative is largest at the boundary and vanishes far from it, so training is driven by cases near the boundary, not by confidently classified ones.</p>

### 9.2 The loss: log-loss

<strong>Maximum likelihood</strong> replaces least squares: choose the parameters under which the observed labels were most probable. The training set had probability \(\prod_i p_i^{y_i}(1 - p_i)^{1 - y_i}\); one minimises its negative logarithm, the <strong>log-loss</strong> (binary cross-entropy):

\[ L(\boldsymbol\beta) = -\frac{1}{n}\sum_{i=1}^{n}\Big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\Big], \qquad p_i = \sigma(\mathbf{x}_i^\top\boldsymbol\beta) \]

A benign case (\(y_i = 1\)) predicted at \(p_i = 0.9\) contributes \(-\log 0.9 \approx 0.105\); at \(p_i = 0.1\), about 2.3; as \(p_i \to 0\), an unbounded penalty. Confident wrong predictions are punished without limit, which keeps the probabilities honest. Its gradient has the form of Section 7.3 — \(\nabla L = \frac{1}{n}\mathbf{X}^\top(\mathbf{p} - \mathbf{y})\), features times residuals — so gradient descent applies unchanged (Exercise 4), and the loss is convex. But the sigmoid is not linear, so there is no closed form: the first model whose training <em>requires</em> an optimiser (R's <code>glm</code> uses iteratively reweighted least squares).

### 9.3 Fitting the model on two features

The chunk fits the model to clump thickness and cell size on the training split of Figure 4. The positive class is benign (code 1).



In [ ]:
# Logistic regression on two features: coefficients, log-odds interpretation, accuracy and log-loss.
dlog <- data.frame(thickness = bc$Cl.thickness, size = bc$Cell.size, y = y)
glm.fit <- glm(y ~ thickness + size, binomial, dlog[tr, ])
cf <- coef(glm.fit)
cat(sprintf("log-odds(benign) = %.2f + (%.3f) x thickness + (%.3f) x size\n", cf[1], cf[2], cf[3]))
cat(sprintf("odds ratio per 1 unit of thickness: %.3f | per 1 unit of size: %.3f\n", exp(cf[2]), exp(cf[3])))
p_te <- predict(glm.fit, dlog[-tr, ], type = "response"); y_te <- dlog$y[-tr]
logloss <- function(p, yy) { p <- pmin(pmax(p, 1e-12), 1 - 1e-12); -mean(yy * log(p) + (1 - yy) * log(1 - p)) }
cat(sprintf("test accuracy (threshold 0.5) = %.3f | test log-loss = %.3f | base-rate log-loss = %.3f\n",
            mean((p_te > 0.5) == (y_te == 1)), logloss(p_te, y_te),
            logloss(rep(mean(dlog$y[tr]), length(y_te)), y_te)))



Both coefficients are negative: thicker clumps and larger nuclei lower the log-odds of benignity. Each additional unit of clump thickness multiplies the odds of benignity by about 0.53, roughly halving them, all else equal. Test accuracy at threshold 0.5 is 0.927 — that of the lines of Figure 3, as expected of another straight boundary. New is the log-loss, 0.145 against 0.625 for predicting the base rate: how much better the probabilities are, which accuracy cannot see.

### 9.4 Worked example by hand: one probability

Round the coefficients to \(\beta_0 = 7.32\), \(\beta_{\text{thickness}} = -0.63\), \(\beta_{\text{size}} = -1.23\) and take a new aspirate with clump thickness 5 and cell size 3.

**Step 1 — score.** \(z = 7.32 - 0.63 \times 5 - 1.23 \times 3 = 7.32 - 3.15 - 3.69 = 0.48\).

**Step 2 — sigmoid.** \(e^{-0.48} \approx 0.619\), so \(p(\text{benign}) = 1/(1 + 0.619) \approx 0.618\) and \(p(\text{malignant}) \approx 0.382\).

**Step 3 — decision.** At threshold 0.5 the case is benign — but the model says nearly two chances in five of malignancy, a statement a clinician would act on. A second case, thickness 8 and size 8: \(z = -7.56\), \(p(\text{benign}) \approx 0.001\).



In [ ]:
# Confirming the hand calculation with the fitted model (unrounded) and with the rounded coefficients.
new_cases <- data.frame(thickness = c(5, 8), size = c(3, 8))
z_hand <- 7.32 - 0.63 * new_cases$thickness - 1.23 * new_cases$size
cat("rounded coefficients : z =", round(z_hand, 3), " p(benign) =", round(sigmoid(z_hand), 3), "\n")
cat("fitted model         : z =", round(predict(glm.fit, new_cases), 3),
    " p(benign) =", round(predict(glm.fit, new_cases, type = "response"), 3), "\n")



Rounded and exact coefficients agree to two decimals. The model outputs the probability; the label is a separate step (Section 9.5).



In [ ]:
# The probability surface of the fitted model, its boundary, and the 205 test cases.
gx <- seq(0, 11, length = 200); gy <- seq(0, 11, length = 200)
grid <- expand.grid(thickness = gx, size = gy)
Pg <- matrix(predict(glm.fit, grid, type = "response"), 200)
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
image(gx, gy, Pg, col = hcl.colors(20, "Blue-Red", rev = TRUE), xlab = "clump thickness", ylab = "cell size",
      main = "P(benign): solid line P = 0.5"); contour(gx, gy, Pg, levels = c(0.1, 0.5, 0.9), add = TRUE, lwd = c(.7, 1.6, .7))
points(dlog$thickness[-tr], dlog$size[-tr], pch = 19, cex = .4, col = ifelse(y_te == 1, BLUE, ORANGE))
points(new_cases, pch = 8, col = GREEN, cex = 2)
hist(p_te[y_te == 1], breaks = seq(0, 1, .05), col = adjustcolor(BLUE, .7), xlab = "predicted P(benign)",
     main = "Predicted probabilities by true class", ylim = c(0, 60))
hist(p_te[y_te == 0], breaks = seq(0, 1, .05), col = adjustcolor(ORANGE, .7), add = TRUE)
abline(v = 0.5); legend("top", c("true benign", "true malignant"), fill = c(BLUE, ORANGE), bty = "n", cex = .8)
par(op)



<p class="caption"><b>Figure 12.</b> Left: predicted probability of benignity over the feature plane, the boundary \(P = 0.5\) (solid) and the lines \(P = 0.1\) and \(0.9\). The boundary is straight; the colours add the band of uncertainty where almost all misclassified cases lie. Stars: the hand-worked cases, one at \(P \approx 0.62\), one deep in the red. Right: predicted probabilities of the test cases by true class — most benign near 1, most malignant near 0, and an overlap in the middle where the cases any threshold will misclassify live.</p>

### 9.5 The threshold and the cost of errors

The histogram is the whole decision problem. Wherever the threshold sits, malignant cases to its right are called benign (<strong>false negatives</strong>, missed cancers) and benign cases to its left malignant (<strong>false positives</strong>, unnecessary biopsies); moving it trades one for the other. The default 0.5 is right only if both errors cost the same. If a missed cancer costs ten times a biopsy, the optimal rule calls malignant whenever \(P(\text{malignant}) > 1/11 \approx 0.09\). The chunk computes sensitivity, specificity and total cost under a 10 : 1 ratio for every threshold.



In [ ]:
# Sensitivity, specificity and total cost as functions of the threshold on P(malignant).
p_mal <- 1 - p_te; is_mal <- (y_te == 0); ths <- seq(0.02, 0.98, 0.01)
tab <- data.frame(t(sapply(ths, function(t) {
  call_mal <- p_mal >= t
  tp <- sum(call_mal & is_mal); fn <- sum(!call_mal & is_mal)
  fp <- sum(call_mal & !is_mal); tn <- sum(!call_mal & !is_mal)
  c(threshold = t, sensitivity = tp / sum(is_mal), specificity = tn / sum(!is_mal),
    missed = fn, biopsies = fp, cost_10 = 10 * fn + fp) })))
best <- tab[which.min(tab$cost_10), ]
cat(sprintf("Of %d test cases (%d malignant): lowest 10:1 cost at threshold %.2f (%d missed, %d unnecessary biopsies, cost %d)\n",
            length(y_te), sum(is_mal), best$threshold, best$missed, best$biopsies, best$cost_10))
at05 <- tab[which.min(abs(tab$threshold - 0.5)), ]
cat(sprintf("At threshold 0.5: %d missed, %d unnecessary biopsies\n", at05$missed, at05$biopsies))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(tab$threshold, tab$sensitivity, type = "l", col = ORANGE, lwd = 2, ylim = c(0, 1),
     xlab = "threshold on P(malignant)", ylab = "rate", main = "The two error types trade off")
lines(tab$threshold, tab$specificity, col = BLUE, lwd = 2); abline(v = c(0.5, best$threshold), col = c(GREY, GREEN), lty = c(3, 2))
legend("bottom", c("sensitivity", "specificity"), col = c(ORANGE, BLUE), lty = 1, bty = "n", cex = .8)
plot(tab$threshold, tab$cost_10, type = "l", lwd = 2, xlab = "threshold on P(malignant)",
     ylab = "total cost on 205 test cases", main = "Total cost under two cost ratios")
lines(tab$threshold, tab$missed + tab$biopsies, col = GREY, lty = 2)
abline(v = c(0.5, best$threshold), col = c(GREY, GREEN), lty = c(3, 2))
legend("top", c("10:1 cost", "1:1 cost"), col = c("black", GREY), lty = c(1, 2), bty = "n", cex = .8)
par(op)



<p class="caption"><b>Figure 13.</b> Left: sensitivity and specificity on the 205 test cases as the threshold on \(P(\text{malignant})\) moves from 0 to 1. Right: total cost under two ratios. With equal costs (dashed) the minimum is near 0.5; with a missed cancer weighted ten times a biopsy (solid) it moves far left to about 0.05 (green), where all cancers are caught at the price of extra biopsies, against several missed at 0.5. Nothing about the <em>model</em> changed between the panels; only the decision rule did.</p>

<div class="box key"><h4>Model, probability, decision</h4><p>A probabilistic classifier separates the <em>model</em> (fitted once), the <em>probability</em> (what it knows about a case) and the <em>decision</em> (which depends on costs that are clinical and ethical, not statistical). Decision-curve analysis formalises the threshold choice as net clinical benefit,<sup class="c"><a href="#r28">[28]</a></sup> and calibration — whether cases given \(P = 0.3\) are malignant 30 % of the time — must be checked separately from accuracy and is neglected in most published models.<sup class="c"><a href="#r29">[29]</a></sup> Supplement S1 develops both.</p></div>

### 9.6 Logistic regression in medicine

The risk scores of daily practice are logistic regressions in disguise. The <strong>Framingham risk score</strong> for coronary heart disease was fitted to 5,345 participants followed for 12 years; its point tables are coefficients rounded to integers for computation on paper.<sup class="c"><a href="#r30">[30]</a></sup> <strong>SAPS II</strong> for intensive-care mortality is a logistic regression on 12,997 patients from 137 units, its 17 variables selected and weighted by the regression.<sup class="c"><a href="#r31">[31]</a></sup> The <strong>Gail model</strong> gives a woman's probability of breast cancer from age, reproductive and family history.<sup class="c"><a href="#r32">[32]</a></sup> Each is a linear score in clinical features, a sigmoid and a threshold that is a clinical decision. The TRIPOD statement specifies how such models must be reported — including the full equation — and is the checklist to apply when you read one.<sup class="c"><a href="#r33">[33]</a></sup>

### 9.7 More than two classes

<strong>One-versus-rest</strong> fits \(K\) binary models and picks the most confident. <strong>Multinomial (softmax) regression</strong> fits one score \(z_k = \mathbf{x}^\top\boldsymbol\beta_k\) per class and converts them jointly with the <strong>softmax</strong>:

\[ P(y = k \mid \mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}} \]

positive, summing to one, and reducing to the sigmoid of the score difference for \(K = 2\). The loss is the same log-loss, \(-\frac{1}{n}\sum_i \log P(y = y_i \mid \mathbf{x}_i)\), and the output layer of every neural-network classifier (Units 6 and 8) is a softmax. The Python edition applied it to the UCI handwritten digits<sup class="c"><a href="#r34">[34]</a></sup> (ten classes, 64 pixels); R has no such dataset, so the chunk <em>simulates</em> four classes of 8×8 image patches — a vertical bar, a horizontal bar, a diagonal and a central block — each a noisy template, flattened to 64 pixel features, and fits <code>nnet::multinom</code>.<sup class="c"><a href="#r38">[38]</a></sup>



In [ ]:
# Multinomial (softmax) logistic regression on simulated 8x8 image patches: four classes, 64 pixels.
set.seed(0); amp <- 1.6; sig <- 1.5
gen <- function(cl) { m <- matrix(rnorm(64, 0, sig), 8, 8)
  if (cl == 1) m[, 4:5] <- m[, 4:5] + amp        # vertical bar
  if (cl == 2) m[4:5, ] <- m[4:5, ] + amp        # horizontal bar
  if (cl == 3) for (k in 1:8) m[k, k] <- m[k, k] + amp   # diagonal
  if (cl == 4) m[3:6, 3:6] <- m[3:6, 3:6] + amp  # central block
  as.vector(m) }
N <- 200; lab <- rep(1:4, each = N / 4); Xi <- t(sapply(lab, gen))
set.seed(0); tri <- sample(N, round(0.7 * N))
sm <- multinom(factor(lab[tri]) ~ ., data.frame(Xi[tri, ]), trace = FALSE, decay = 0.2, MaxNWts = 5000)
prob <- predict(sm, data.frame(Xi[-tri, ]), type = "probs"); pr <- predict(sm, data.frame(Xi[-tri, ]))
cat(sprintf("test accuracy on %d patches = %.3f | probabilities sum to one: %s | mean confidence = %.3f\n",
            N - length(tri), mean(pr == lab[-tri]), all(abs(rowSums(prob) - 1) < 1e-8), mean(apply(prob, 1, max))))
cm <- table(true = lab[-tri], predicted = pr)
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
image(1:8, 1:8, matrix(gen(3), 8), col = grey.colors(20), xlab = "", ylab = "", main = "one simulated 'diagonal' patch")
image(1:4, 1:4, t(cm[4:1, ]), col = hcl.colors(12, "Blues", rev = TRUE), xlab = "predicted class",
      ylab = "true class", axes = FALSE, main = "Confusion matrix (60 test patches)")
axis(1, 1:4); axis(2, 1:4, labels = 4:1)
for (i in 1:4) for (j in 1:4) text(j, 5 - i, cm[i, j], col = ifelse(cm[i, j] > max(cm)/2, "white", "black"))
par(op)



<p class="caption"><b>Figure 14.</b> Right: confusion matrix on 60 held-out simulated patches; rows true, columns predicted, diagonal correct. Accuracy is 0.95 — the linear softmax model misreads about one patch in twenty — and the off-diagonal entries show <em>which</em> confusions occur (the diagonal template occasionally read as the central block), information accuracy hides and that matters when some confusions are graver than others. Left: one noisy "diagonal" patch, to show how much noise the model sees through.</p>

<div class="box key"><h4>Exercise 4</h4><p>(a) Implement gradient descent for logistic regression on the two standardised features: from \(\boldsymbol\beta = \mathbf{0}\), gradient \(\frac{1}{n}\mathbf{X}^\top(\sigma(\mathbf{X}\boldsymbol\beta) - \mathbf{y})\), rate 0.5, 500 steps; compare with <code>glm</code> on the same features. (b) Change the cost ratio in the threshold chunk to 3 and to 30. (c) Raise the image noise <code>sig</code> to 2.5 and watch the confusion matrix fill in.</p></div>

## 10. Previews of the other paradigms

### 10.1 Unsupervised: k-means on the breast-cancer features, without labels

<strong>k-means</strong> partitions \(n\) points into \(k\) groups minimising squared distances to group centroids by an alternation worth tracing once by hand: (1) <em>assign</em> each point to the nearest centroid; (2) <em>update</em> each centroid to the mean of its points; repeat until nothing changes (Lloyd, 1957; MacQueen, 1967).<sup class="c"><a href="#r20">[20]</a></sup><sup class="c"><a href="#r19">[19]</a></sup> It is the nearest-centroid classifier of Unit 1 run <em>without labels</em>: it guesses the groups, computes centroids, re-guesses, iterates. The chunk runs it with \(k = 2\) on the two features — never seeing the diagnosis — and only then compares with the labels.



In [ ]:
# k-means (k = 2) on two features of all 683 aspirates, labels hidden; then compared with the labels.
set.seed(0); km <- kmeans(X2, centers = 2, nstart = 10)          # X2 only: no labels
clusters <- km$cluster - 1
agreement <- max(mean(clusters == yv), mean(clusters != yv))     # cluster numbering is arbitrary
ct <- table(cluster = clusters, biopsy = ifelse(yv == 0, "malignant", "benign"))
cat("centroids found without labels:\n"); print(round(km$centers, 2))
cat(sprintf("agreement between clusters and diagnosis (best label matching): %.3f = %d of %d\n",
            agreement, round(agreement * nrow(bc)), nrow(bc)))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
plot(X2, col = ifelse(clusters == round(mean(clusters[yv == 1])), BLUE, ORANGE), pch = 19, cex = .4,
     xlab = "clump thickness", ylab = "cell size", main = "What k-means found (no labels)")
points(km$centers, pch = 4, cex = 2, lwd = 3)
plot(X2, col = ifelse(yv == 1, BLUE, ORANGE), pch = 19, cex = .4, xlab = "clump thickness", ylab = "cell size",
     main = "The biopsy labels (blue benign)")
par(op); print(ct)



<p class="caption"><b>Figure 15.</b> Left: the two groups found from the features alone, with centroids (crosses). Right: the same points coloured by biopsy. The unsupervised algorithm recovered a partition agreeing with the pathologist for about 93 % of the aspirates (633 of 683), because the two diagnoses happen to form two dense regions and k-means finds dense regions; the cross-tabulation shows the cases on the wrong side of the gap.</p>

Encouraging and a warning. Structure in unlabelled data can coincide with clinical categories — the hope behind every search for patient subtypes. But the agreement is near the supervised methods only because the classes separate cleanly here; the algorithm was told that <em>two</em> groups existed, and had the natural clusters been "large" and "small" tumours it would have found those and nothing in its output would have said so. Unit 5 treats the choice of \(k\) and the validation of a clustering.

### 10.2 Reinforcement: a five-state corridor

Positions 0 to 4; the agent starts at 2 and moves left or right. Position 0 is an exit with reward 0.3, position 4 an exit with reward 1.0; both end the episode; other moves earn nothing. One step from the small exit the agent faces the defining question: take 0.3 now, or walk three steps for 1.0? The answer depends on the <strong>discount factor</strong> \(\gamma \in [0, 1)\): a reward \(t\) steps away is worth \(\gamma^t\) of one now. The <strong>value</strong> \(V^*(s)\), the best discounted sum of future rewards from \(s\), satisfies Bellman's equation:

\[ V^*(s) = \max_{a} \Big[\, r(s, a) + \gamma\, V^*(s') \,\Big] \]

where \(s'\) is the state reached by action \(a\) and \(V^*\) of an exit is 0. <strong>Value iteration</strong> starts with \(V = 0\) and applies the right-hand side repeatedly until \(V\) stops changing; the optimal policy takes, in each state, the maximising action.<sup class="c"><a href="#r11">[11]</a></sup>

**Worked by hand, \(\gamma = 0.9\).** Positions 1, 2, 3 have values to compute; start with \(V = (0, 0, 0)\).

| sweep | \(V(1) = \max(0.3,\; 0.9\,V(2))\) | \(V(2) = \max(0.9\,V(1),\; 0.9\,V(3))\) | \(V(3) = \max(0.9\,V(2),\; 1.0)\) |
|---|---|---|---|
| 0 | 0 | 0 | 0 |
| 1 | \(\max(0.3, 0) = 0.3\) | \(\max(0, 0) = 0\) | \(\max(0, 1.0) = 1.0\) |
| 2 | \(\max(0.3, 0) = 0.3\) | \(\max(0.27, 0.9) = 0.9\) | \(1.0\) |
| 3 | \(\max(0.3, 0.81) = 0.81\) | \(0.9\) | \(1.0\) |
| 4 | \(0.81\) | \(0.9\) | \(1.0\) — converged |

\(V^* = (0.81, 0.9, 1.0)\) and the maximising action is "right" everywhere: the agent ignores the small exit even when next to it, because \(0.9^2 = 0.81 > 0.3\). With \(\gamma = 0.5\), position 1 gives \(\max(0.3, 0.25) = 0.3\) and the agent takes the small exit — a myopic agent settles for less.



In [ ]:
# Value iteration on the five-state corridor, for two discount factors.
step <- function(s, a) { s2 <- s + a                               # move; returns (new state, reward, ended)
  if (s2 == 0) return(list(s2, 0.3, TRUE)); if (s2 == 4) return(list(s2, 1.0, TRUE)); list(s2, 0, FALSE) }
value_iteration <- function(gamma, tol = 1e-9) {
  V <- rep(0, 5); sweeps <- 0
  repeat { Vn <- V
    for (s in c(1, 2, 3)) Vn[s + 1] <- max(sapply(c(-1, 1), function(a) {
      r <- step(s, a); r[[2]] + gamma * (if (r[[3]]) 0 else V[r[[1]] + 1]) }))
    sweeps <- sweeps + 1; if (max(abs(Vn - V)) < tol) return(list(V = Vn, sweeps = sweeps)); V <- Vn } }
for (gamma in c(0.9, 0.5)) { out <- value_iteration(gamma)
  policy <- sapply(c(1, 2, 3), function(s) {
    rR <- step(s, 1); rL <- step(s, -1)
    vR <- rR[[2]] + gamma * (if (rR[[3]]) 0 else out$V[rR[[1]] + 1])
    vL <- rL[[2]] + gamma * (if (rL[[3]]) 0 else out$V[rL[[1]] + 1]); if (vR >= vL) "right" else "left" })
  cat(sprintf("gamma = %.1f: V*(1..3) = %s after %d sweeps | policy at 1,2,3: %s\n",
              gamma, paste(round(out$V[2:4], 3), collapse = ", "), out$sweeps, paste(policy, collapse = ", "))) }



Value iteration reproduces \(V^* = (0.81, 0.9, 1.0)\) for \(\gamma = 0.9\); for \(\gamma = 0.5\) the agent at position 1 turns left. But it needs the rules of the environment — <code>step</code> — in advance. <strong>Q-learning</strong> learns a table \(Q(s, a)\) from experience alone, trying actions (exploring at random with probability \(\epsilon\)) and updating

\[ Q(s, a) \leftarrow Q(s, a) + \alpha\,\Big[\, r + \gamma \max_{a'} Q(s', a') - Q(s, a) \,\Big], \]

where the bracket is the <em>temporal-difference error</em> between expectation and observation and \(\alpha\) is a learning rate as in gradient descent. Watkins and Dayan proved that the table converges to \(Q^*\), with \(V^*(s) = \max_a Q^*(s, a)\).<sup class="c"><a href="#r35">[35]</a></sup>



In [ ]:
# Q-learning: the same values learned from experience only, without access to the rules.
set.seed(0); gamma <- 0.9; alpha <- 0.5; epsilon <- 0.3
Q <- matrix(0, 5, 2); V_star <- value_iteration(gamma)$V; errors <- numeric(600)
for (episode in 1:600) { s <- 2                                   # every episode starts in the middle
  for (t in 1:50) {
    a <- if (runif(1) < epsilon) sample(2, 1) else which.max(Q[s + 1, ])   # explore or exploit
    r <- step(s, c(-1, 1)[a]); target <- if (r[[3]]) r[[2]] else r[[2]] + gamma * max(Q[r[[1]] + 1, ])
    Q[s + 1, a] <- Q[s + 1, a] + alpha * (target - Q[s + 1, a]); s <- r[[1]]; if (r[[3]]) break }
  errors[episode] <- max(abs(apply(Q[2:4, ], 1, max) - V_star[2:4])) }
cat("learned Q(s,a) for positions 1..3 [left, right]:\n"); print(round(Q[2:4, ], 3))
cat("V from Q =", round(apply(Q[2:4, ], 1, max), 3), "| value iteration =", round(V_star[2:4], 3), "\n")
plot(errors, type = "l", col = BLUE, log = "y", xlab = "episode", ylab = "max |V from Q - V*|",
     main = "Q-learning converges to the values from value iteration")



<p class="caption"><b>Figure 16.</b> Largest error between the values implied by the Q-table and \(V^*\) over 600 episodes. By trial and error alone the agent discovers the values that value iteration computed from the rules — and therefore the same policy: in every position the "right" entry of the table exceeds the "left" one.</p>

Nothing in the Q-learning chunk knows the rewards or transitions; it calls <code>step</code> and observes. That is the sense in which "reinforcement" names a paradigm: the experience is generated by the learner's actions, the performance is cumulative reward, the task is a policy. Scale the corridor to a patient's state in intensive care, the actions to doses and the reward to survival, and you have the "AI Clinician" of Section 3.4<sup class="c"><a href="#r12">[12]</a></sup> — with its difficulties: experience taken from records of what clinicians did, a state never fully observed, and a reward as delayed as feedback gets.

<div class="box key"><h4>Exercise 5</h4><p>(a) Set <code>epsilon = 0</code> in the Q-learning chunk. Does the agent still find the right exit? Explain with the first episodes in mind. (b) Set the left reward to 0.85 with \(\gamma = 0.9\). Which exit should the agent take from positions 1 and 2? Confirm with value iteration.</p></div>

## 11. Preview: how we will know a model works

Every experiment above measured a model on unseen data, and Section 8 showed why. Supplement S1 develops the methodology; this section fixes its vocabulary.

**Training, validation and test sets.** The labelled data are split at random into a <strong>training set</strong>, on which parameters are fitted; a <strong>validation set</strong>, on which flexibility knobs (degree, \(\lambda\), tree depth, the threshold of Section 9.5) are chosen by comparing candidates; and a <strong>test set</strong>, touched once at the end to report the performance of the final model on data that influenced none of its decisions. Usual proportions are 60/20/20, 70/15/15 and 80/10/10 per cent, <em>stratified</em> to keep class proportions. With few data, <strong>k-fold cross-validation</strong> replaces the validation set: divide the training data into \(k\) folds (5 or 10), fit \(k\) times on \(k - 1\) folds, evaluate on the remaining one, average; stratified k-fold keeps class proportions, leave-one-out is \(k = n\).

**Overfitting** is a model that has learned the accidents of its sample — noise, outliers, spurious correlations — besides the pattern: low training error, high error on new data, as for the degree-15 polynomial; it grows with flexibility, fewer cases and noisier labels, and is detected only on unseen data. **Underfitting** is a model too rigid for the pattern, with high error on both, as for the line fitted to the sine. The bias–variance decomposition of Supplement S1 explains why generalisation error is U-shaped between them.

**Metrics for a classifier.** The <strong>confusion matrix</strong> (Figure 14) tabulates true against predicted classes; for two classes its cells are true positives (TP), false positives (FP), false negatives (FN), true negatives (TN). <strong>Accuracy</strong> = (TP + TN)/n, misleading with imbalanced classes; <strong>precision</strong> = TP/(TP + FP), the fraction of positive calls that are right; <strong>recall</strong> (sensitivity) = TP/(TP + FN), the fraction of true positives caught; <strong>specificity</strong> = TN/(TN + FP); <strong>F1</strong>, the harmonic mean of precision and recall; <strong>AUC-ROC</strong>, the area under the curve of sensitivity against 1 − specificity as the threshold moves (the two curves of Figure 13 plotted against each other), measuring how well the model ranks cases independently of any threshold — 0.5 for a coin, 1.0 for a perfect ranking. For regression the analogues are MSE, RMSE and \(R^2\) on held-out data. Imbalanced classes, thresholds, calibration, confidence intervals and information leakage between splits are developed in Supplement S1, to be read before Unit 3.

## 12. Summary

You should now be able to explain, without notes:

- Mitchell's \(T\), \(P\), \(E\); learning as induction and what that implies about certainty.
- The seven steps of a data-science project and why machine learning is one of them.
- Model, loss, optimiser; the paradigms by their feedback (supervised, unsupervised with its four sub-types, reinforcement, semi- and self-supervised); class versus target.
- The statistical, neural and evolutionary approaches as searches of one hypothesis space; classifier and decision boundary; why k-means is not a classifier.
- Least squares, the normal equations \(\mathbf{X}^\top\mathbf{X}\boldsymbol\beta = \mathbf{X}^\top\mathbf{y}\), the four-point fit by hand, coefficients, \(R^2\), residuals.
- Gradient descent: update, learning-rate regimes, conditioning and scaling, stochastic variants.
- Overfitting, underfitting and what ridge adds to the normal equations.
- Logistic regression: sigmoid, log-odds, log-loss, one probability by hand, threshold as a decision.
- What k-means, value iteration and Q-learning compute and from what experience.
- Training/validation/test, cross-validation, confusion matrix, accuracy, precision, recall, F1, AUC.

## 13. Self-assessment

<details><summary>1. A hospital wants to flag, at admission, patients likely to need intensive care within 48 hours from their first vital signs and laboratory results. Identify \(T\), \(P\) and \(E\) and the paradigm.</summary>\(T\): assign "ICU within 48 h" or not to an admission from its initial measurements; \(P\): e.g. sensitivity at a fixed false-alarm rate; \(E\): past admissions with outcome. Supervised classification.</details>

<details><summary>2. Inductive or deductive: (a) "The dosing calculator applies the weight-based rule to this patient"; (b) "In 2,000 past patients this dose was toxic when creatinine clearance was below 30 mL/min, so the rule should include a renal adjustment"?</summary>(a) Deduction: a known rule applied to a case. (b) Induction: a general rule proposed from cases, possibly wrong for cases unlike those observed.</details>

<details><summary>3. "The data are already in the database, so we can start fitting models tomorrow." Which steps of the protocol are skipped, and what could go wrong in each?</summary>Problem identification (undefined target: the wrong question answered), cleaning (missing values coded as zeros, duplicated patients, identifiers as features) and exploration (a feature that is a proxy for the label, typing errors as extreme values).</details>

<details><summary>4. In Figure 3 two methods produced similar lines, and LDA a third close to them. Are the three approaches therefore equivalent?</summary>No. They shared the hypothesis space (lines), so none could find anything else; they differ in how they search, in what else they provide and, with richer spaces, in what they can represent.</details>

<details><summary>5. Fit a least-squares line by hand to \((1, 2), (2, 3), (3, 5), (4, 6)\).</summary>\(\bar{x} = 2.5\), \(\bar{y} = 4\). Products of deviations: \(3 + 0.5 + 0.5 + 3 = 7\); \(\sum (x - \bar{x})^2 = 5\). Slope \(1.4\), intercept \(4 - 1.4 \times 2.5 = 0.5\): \(\hat{y} = 0.5 + 1.4x\); residuals \(0.1, -0.3, 0.3, -0.1\), summing to zero.</details>

<details><summary>6. The rooms coefficient was 9.1 alone and 3.8 with thirteen features. Which is "the effect of rooms"?</summary>Neither, causally. The first is the average difference per room; the second the difference among tracts equal on the other twelve variables, which absorb part of the association.</details>

<details><summary>7. Gradient descent diverged at rate 0.021 and converged at 0.015. What sets the boundary, and what change to the data moves it to 1.0?</summary>The largest Hessian eigenvalue, \(\lambda_{\max} \approx 102\): convergence needs \(\eta < 2/\lambda_{\max} \approx 0.0196\). Standardising the feature makes the Hessian of the MSE \(2\mathbf{I}\), so the boundary becomes \(\eta < 1\).</details>

<details><summary>8. In Figure 9 degree 15 has the lowest training and the highest test error. Explain both, and what ridge changed in Figure 10.</summary>Sixteen parameters pass close to twenty points (small training error); between them the curve follows noise (large test error). Ridge added \(\lambda\mathbf{I}\) to \(\mathbf{X}^\top\mathbf{X}\), shrinking the coefficients by orders of magnitude and forcing a smooth curve.</details>

<details><summary>9. With \(z = 7.32 - 0.63\,t - 1.23\,s\) (thickness \(t\), size \(s\)), compute \(P(\text{benign})\) for thickness 6 and size 2, and the odds ratio per unit of \(s\).</summary>\(z = 7.32 - 3.78 - 2.46 = 1.08\); \(P = 1/(1 + e^{-1.08}) \approx 0.746\), so \(P(\text{malignant}) \approx 0.254\). Odds ratio \(e^{-1.23} \approx 0.29\) per unit of cell size.</details>

<details><summary>10. Why did the optimal threshold in Figure 13 fall far below 0.5 under a 10 : 1 cost ratio, and what is the cost-optimal threshold in general?</summary>Lowering the threshold converts expensive false negatives into cheap false positives. Calling a case malignant is cheaper in expectation whenever \(P(\text{malignant}) > c_{FP}/(c_{FP} + c_{FN})\); for 10 : 1, \(1/11 \approx 0.09\) (the empirical minimum on the test set was about 0.05).</details>

<details><summary>11. k-means agreed with the biopsy for 93 % of aspirates without labels. A colleague concludes the labels were unnecessary. What is wrong?</summary>The agreement could only be measured <em>because</em> labels existed; without them nothing says the clusters mean diagnosis rather than size. k-means was also told \(k = 2\), and the supervised methods were at least as good with a probability attached.</details>

<details><summary>12. In the corridor with \(\gamma = 0.9\), compute \(V(1)\) if the right exit paid 0.5 instead of 1.0, and the policy at position 1.</summary>\(V(3) = 0.5\), \(V(2) = 0.45\), \(V(1) = \max(0.3, 0.405) = 0.405\): still right, barely; with \(\gamma = 0.75\), \(0.28 < 0.3\) and it turns left.</details>

## Glossary

<dl class="gloss">
<dt>Classifier / decision boundary</dt><dd>A function from feature vectors to class labels / the surface where the predicted class changes.</dd>
<dt>Coefficient of determination (R²)</dt><dd>\(1 - \text{RSS}/\text{TSS}\): the fraction of target variance accounted for by the model.</dd>
<dt>Cross-validation (k-fold)</dt><dd>Fitting \(k\) times, each holding out one of \(k\) folds, and averaging the scores.</dd>
<dt>Curse of dimensionality</dt><dd>Degradation of performance as features are added at fixed sample size, because data become sparse.</dd>
<dt>Deductive / inductive inference</dt><dd>From rules to cases (certain, rigid) / from cases to a rule (probable, generalising); learning is inductive.</dd>
<dt>Discount factor (γ)</dt><dd>The factor by which a reward one step in the future is devalued relative to an immediate one.</dd>
<dt>Gradient descent</dt><dd>\(\theta \leftarrow \theta - \eta\nabla L(\theta)\); stochastic gradient descent estimates the gradient from a mini-batch.</dd>
<dt>IC50 / pIC50</dt><dd>Concentration inhibiting a target's activity by half / its negative decimal logarithm in mol/L, the usual QSAR target.</dd>
<dt>Learning rate (η)</dt><dd>The step size of gradient descent; too small is slow, too large diverges.</dd>
<dt>Least squares / normal equations</dt><dd>Minimising the sum of squared residuals / the linear system \(\mathbf{X}^\top\mathbf{X}\boldsymbol\beta = \mathbf{X}^\top\mathbf{y}\) that solves it.</dd>
<dt>Log-loss (cross-entropy)</dt><dd>The negative mean log-probability assigned to the observed labels; the loss of logistic regression and of neural classifiers.</dd>
<dt>Logistic regression</dt><dd>\(P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{x}^\top\boldsymbol\beta)\) fitted by maximum likelihood; coefficients are log-odds ratios.</dd>
<dt>Overfitting / underfitting</dt><dd>Fitting the noise of the sample (low training, high test error) / being too rigid for the pattern (high error on both).</dd>
<dt>Policy / value function</dt><dd>A mapping from states to actions / the expected discounted cumulative reward from a state.</dd>
<dt>Regression / classification</dt><dd>Supervised learning with a continuous target / with a categorical target (class, label).</dd>
<dt>Regularisation (ridge)</dt><dd>A penalty on parameter size added to the loss; ridge uses \(\lambda\lVert\boldsymbol\beta\rVert^2\), i.e. \(\lambda\mathbf{I}\) in the normal equations.</dd>
<dt>Sigmoid / softmax</dt><dd>\(\sigma(z) = 1/(1 + e^{-z})\), mapping a score to a probability (inverse: logit) / its \(K\)-class generalisation.</dd>
<dt>Structured / unstructured data</dt><dd>Data with a predefined model (tables) / without one (text, images, signals), converted by feature extraction.</dd>
<dt>Supervised / unsupervised / reinforcement learning</dt><dd>Learning from labelled pairs / from inputs alone / from rewards obtained by acting.</dd>
<dt>Training / validation / test set</dt><dd>Data used to fit parameters / to choose flexibility settings / to report final performance once.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Mitchell TM. <em>Machine Learning</em>. New York: McGraw-Hill; 1997. Chapter 1. <a href="http://www.cs.cmu.edu/~tom/mlbook.html">www.cs.cmu.edu/~tom/mlbook.html</a> (book; no DOI — publisher/author page)</li>
<li id="r2">Samuel AL. Some studies in machine learning using the game of checkers. <em>IBM J Res Dev</em>. 1959;3(3):210–229. <a href="https://doi.org/10.1147/rd.33.0210">doi:10.1147/rd.33.0210</a></li>
<li id="r3">Domingos P. A few useful things to know about machine learning. <em>Commun ACM</em>. 2012;55(10):78–87. <a href="https://doi.org/10.1145/2347736.2347755">doi:10.1145/2347736.2347755</a></li>
<li id="r4">Donoho D. 50 years of data science. <em>J Comput Graph Stat</em>. 2017;26(4):745–766. <a href="https://doi.org/10.1080/10618600.2017.1384734">doi:10.1080/10618600.2017.1384734</a></li>
<li id="r5">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r6">Lipinski CA, Lombardo F, Dominy BW, Feeney PJ. Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. <em>Adv Drug Deliv Rev</em>. 2001;46(1–3):3–26. <a href="https://doi.org/10.1016/S0169-409X(00)00129-0">doi:10.1016/S0169-409X(00)00129-0</a></li>
<li id="r7">Wishart DS, Feunang YD, Guo AC, et al. DrugBank 5.0: a major update to the DrugBank database for 2018. <em>Nucleic Acids Res</em>. 2018;46(D1):D1074–D1082. <a href="https://doi.org/10.1093/nar/gkx1037">doi:10.1093/nar/gkx1037</a></li>
<li id="r8">Mendez D, Gaulton A, Bento AP, et al. ChEMBL: towards direct deposition of bioassay data. <em>Nucleic Acids Res</em>. 2019;47(D1):D930–D940. <a href="https://doi.org/10.1093/nar/gky1075">doi:10.1093/nar/gky1075</a></li>
<li id="r9">Hughes GF. On the mean accuracy of statistical pattern recognizers. <em>IEEE Trans Inf Theory</em>. 1968;14(1):55–63. <a href="https://doi.org/10.1109/TIT.1968.1054102">doi:10.1109/TIT.1968.1054102</a></li>
<li id="r10">Agrawal R, Imieliński T, Swami A. Mining association rules between sets of items in large databases. <em>ACM SIGMOD Rec</em>. 1993;22(2):207–216. <a href="https://doi.org/10.1145/170036.170072">doi:10.1145/170036.170072</a></li>
<li id="r11">Sutton RS, Barto AG. <em>Reinforcement Learning: An Introduction</em>. 2nd ed. Cambridge, MA: MIT Press; 2018. <a href="http://incompleteideas.net/book/the-book-2nd.html">incompleteideas.net/book/the-book-2nd.html</a> (book; no DOI — authors' page with the full text)</li>
<li id="r12">Komorowski M, Celi LA, Badawi O, Gordon AC, Faisal AA. The Artificial Intelligence Clinician learns optimal treatment strategies for sepsis in intensive care. <em>Nat Med</em>. 2018;24(11):1716–1720. <a href="https://doi.org/10.1038/s41591-018-0213-5">doi:10.1038/s41591-018-0213-5</a></li>
<li id="r13">Chapelle O, Schölkopf B, Zien A, eds. <em>Semi-Supervised Learning</em>. Cambridge, MA: MIT Press; 2006. <a href="https://doi.org/10.7551/mitpress/9780262033589.001.0001">doi:10.7551/mitpress/9780262033589.001.0001</a></li>
<li id="r14">Krishnan R, Rajpurkar P, Topol EJ. Self-supervised learning in medicine and healthcare. <em>Nat Biomed Eng</em>. 2022;6(12):1346–1352. <a href="https://doi.org/10.1038/s41551-022-00914-1">doi:10.1038/s41551-022-00914-1</a></li>
<li id="r15">Jain AK, Duin RPW, Mao J. Statistical pattern recognition: a review. <em>IEEE Trans Pattern Anal Mach Intell</em>. 2000;22(1):4–37. <a href="https://doi.org/10.1109/34.824819">doi:10.1109/34.824819</a></li>
<li id="r16">Rosenblatt F. The perceptron: a probabilistic model for information storage and organization in the brain. <em>Psychol Rev</em>. 1958;65(6):386–408. <a href="https://doi.org/10.1037/h0042519">doi:10.1037/h0042519</a></li>
<li id="r17">Holland JH. Genetic algorithms. <em>Sci Am</em>. 1992;267(1):66–72. <a href="https://doi.org/10.1038/scientificamerican0792-66">doi:10.1038/scientificamerican0792-66</a></li>
<li id="r18">Hand DJ. Classifier technology and the illusion of progress. <em>Stat Sci</em>. 2006;21(1):1–14. <a href="https://doi.org/10.1214/088342306000000060">doi:10.1214/088342306000000060</a></li>
<li id="r19">MacQueen J. Some methods for classification and analysis of multivariate observations. In: <em>Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability</em>, vol. 1. Berkeley: University of California Press; 1967:281–297. <a href="https://projecteuclid.org/euclid.bsmsp/1200512992">projecteuclid.org/euclid.bsmsp/1200512992</a> (no DOI — Project Euclid landing page)</li>
<li id="r20">Lloyd SP. Least squares quantization in PCM. <em>IEEE Trans Inf Theory</em>. 1982;28(2):129–137. <a href="https://doi.org/10.1109/TIT.1982.1056489">doi:10.1109/TIT.1982.1056489</a></li>
<li id="r21">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r22">Galton F. Regression towards mediocrity in hereditary stature. <em>J Anthropol Inst G B Irel</em>. 1886;15:246–263. <a href="https://doi.org/10.2307/2841583">doi:10.2307/2841583</a></li>
<li id="r23">Stigler SM. Gauss and the invention of least squares. <em>Ann Stat</em>. 1981;9(3):465–474. <a href="https://doi.org/10.1214/aos/1176345451">doi:10.1214/aos/1176345451</a></li>
<li id="r24">Hoerl AE, Kennard RW. Ridge regression: biased estimation for nonorthogonal problems. <em>Technometrics</em>. 1970;12(1):55–67. <a href="https://doi.org/10.1080/00401706.1970.10488634">doi:10.1080/00401706.1970.10488634</a></li>
<li id="r25">Robbins H, Monro S. A stochastic approximation method. <em>Ann Math Stat</em>. 1951;22(3):400–407. <a href="https://doi.org/10.1214/aoms/1177729586">doi:10.1214/aoms/1177729586</a></li>
<li id="r26">Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning: Data Mining, Inference, and Prediction</em>. 2nd ed. New York: Springer; 2009. <a href="https://doi.org/10.1007/978-0-387-84858-7">doi:10.1007/978-0-387-84858-7</a></li>
<li id="r27">Cox DR. The regression analysis of binary sequences. <em>J R Stat Soc Ser B</em>. 1958;20(2):215–232. <a href="https://doi.org/10.1111/j.2517-6161.1958.tb00292.x">doi:10.1111/j.2517-6161.1958.tb00292.x</a></li>
<li id="r28">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565–574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r29">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r30">Wilson PWF, D'Agostino RB, Levy D, Belanger AM, Silbershatz H, Kannel WB. Prediction of coronary heart disease using risk factor categories. <em>Circulation</em>. 1998;97(18):1837–1847. <a href="https://doi.org/10.1161/01.CIR.97.18.1837">doi:10.1161/01.CIR.97.18.1837</a></li>
<li id="r31">Le Gall J-R, Lemeshow S, Saulnier F. A new Simplified Acute Physiology Score (SAPS II) based on a European/North American multicenter study. <em>JAMA</em>. 1993;270(24):2957–2963. <a href="https://doi.org/10.1001/jama.1993.03510240069035">doi:10.1001/jama.1993.03510240069035</a></li>
<li id="r32">Gail MH, Brinton LA, Byar DP, et al. Projecting individualized probabilities of developing breast cancer for white females who are being examined annually. <em>J Natl Cancer Inst</em>. 1989;81(24):1879–1886. <a href="https://doi.org/10.1093/jnci/81.24.1879">doi:10.1093/jnci/81.24.1879</a></li>
<li id="r33">Collins GS, Reitsma JB, Altman DG, Moons KGM. Transparent Reporting of a multivariable prediction model for Individual Prognosis Or Diagnosis (TRIPOD): the TRIPOD statement. <em>Ann Intern Med</em>. 2015;162(1):55–63. <a href="https://doi.org/10.7326/M14-0697">doi:10.7326/M14-0697</a></li>
<li id="r34">Alpaydin E, Kaynak C. Optical Recognition of Handwritten Digits [dataset]. UCI Machine Learning Repository; 1998. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r35">Watkins CJCH, Dayan P. Q-learning. <em>Mach Learn</em>. 1992;8(3–4):279–292. <a href="https://doi.org/10.1007/BF00992698">doi:10.1007/BF00992698</a></li>
<li id="r36">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; no DOI)</li>
<li id="r37">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r38">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. New York: Springer; 2002 (the <code>MASS</code>, <code>class</code> and <code>nnet</code> packages). <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r39">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. New York: Springer; 2021. <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a></li>
<li id="r40">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Mitchell TM. <em>Machine Learning</em>. McGraw-Hill; 1997. — Chapter 1 for the definition of learning; chapters 4 and 13 for gradient descent and reinforcement learning.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Chapters 2–4 cover Sections 6–9 with R code; free PDF at statlearning.com.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapters 3–4 for linear and logistic regression and ridge; free PDF.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. — the reference for the R modelling functions used here (<code>lda</code>, <code>nnet</code>, <code>multinom</code>).</li>
<li>Sutton RS, Barto AG. <em>Reinforcement Learning: An Introduction</em>. 2nd ed. MIT Press; 2018. — Chapters 3–4 and 6 for value iteration and Q-learning; free online.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — Chapter 1 for the polynomial experiment of Section 8 in its original form.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>CRAN task view "Machine Learning &amp; Statistical Learning" — <a href="https://cran.r-project.org/view=MachineLearning">cran.r-project.org/view=MachineLearning</a>.</li>
<li>ChEMBL — <a href="https://www.ebi.ac.uk/chembl/">ebi.ac.uk/chembl</a>; BindingDB — <a href="https://www.bindingdb.org">bindingdb.org</a>; DrugBank — <a href="https://go.drugbank.com">go.drugbank.com</a>; Drug Repurposing Hub — <a href="https://clue.io/repurposing">clue.io/repurposing</a>.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a> (the original Wisconsin breast-cancer and Boston housing data).</li>
<li>Framingham Heart Study risk functions — <a href="https://www.framinghamheartstudy.org/fhs-risk-functions/">framinghamheartstudy.org/fhs-risk-functions</a>.</li>
</ul>
